# Container Resource Prediction — Full Pipeline
## Phase 1: Data Preprocessing + Sequence Generation
## Phase 2: GRU Model Architecture & Data Pipeline

---

### Pipeline Overview
```
Phase 1 — Preprocessing
  Step 1+2 : Load raw CSVs from Google Drive  →  Pivot long to wide format
  Step 3a  : Chronological split (60/20/20)   →  Normalize (train stats only)
  Step 3b  : Feature engineering              →  Lag diffs + rolling features
  Step 4   : Sliding window sequences         →  .npy files per horizon

Phase 2 — GRU Model Architecture
  Step 5   : SequenceDataset                  →  Wraps .npy as PyTorch Dataset
  Step 6   : DataLoaders                      →  Configurable batching
  Step 7   : GRUModel                         →  2-layer GRU -> FC -> Output
  Step 8   : Smoke test                       →  Verify shapes with real data
```

---
**Author:** Team-Dracasys | **Version:** Phase 1 + Phase 2 Combined

# STEP 0: Memory Monitoring Setup

In [46]:
import psutil
import os
import gc

def get_memory_usage():
    """Get current memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def log_memory(label: str):
    """Log memory usage with label."""
    gc.collect()  # Force garbage collection
    mem = get_memory_usage()
    print(f"💾 [{label}] RAM: {mem:.1f} MB")
    return mem

print("✓ Memory monitoring setup complete")
initial_mem = log_memory("Initial")

✓ Memory monitoring setup complete
💾 [Initial] RAM: 8435.9 MB


# STEP 1: Mount Google Drive & Verify Access

In [47]:
import sys
import os

# Detect the runtime via filesystem markers, NOT package importability.
# Kaggle's Docker image ships the google-colab package too (it imports fine
# there), so "try: import google.colab" alone gives a FALSE POSITIVE for
# IN_COLAB on Kaggle -- that false positive is exactly what caused
# drive.mount() to be called on Kaggle and fail with NotImplementedError.
# /var/colab/hostname only exists on a genuine Colab VM -- it's the same check
# google.colab.drive.mount() uses internally, so this can't give a false
# positive the way the import-based check did.
IN_COLAB  = os.path.exists('/var/colab/hostname')
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    print("✓ Running in Google Colab")
elif IN_KAGGLE:
    print("✓ Running in Kaggle")
else:
    print("⚠ Running locally (not Google Colab, not Kaggle)")

print(f"Python version: {sys.version}")

✓ Running in Kaggle
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [48]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive

    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    print("\n✓ Google Drive mounted!")

    raw_path = Path('/content/drive/My Drive/raw')

    if raw_path.exists():
        print(f"✓ Found raw folder at: {raw_path}")
        all_csv_files = list(raw_path.glob('**/*.csv'))
        print(f"✓ Found {len(all_csv_files)} CSV files")
    else:
        print(f"❌ raw folder not found at: {raw_path}")

elif IN_KAGGLE:
    # Kaggle can't mount Google Drive -- drive.mount() is Colab-only. gdown's
    # folder download (the earlier approach here) turned out unreliable for
    # this dataset: it resolves a public link per FILE (~109 individual CSVs),
    # and kept failing/rate-limiting on individual files even with the parent
    # folder shared correctly. Switched to a Kaggle Dataset instead -- fully
    # offline in the kernel once attached, no Drive permission/rate-limit
    # class of problem at all.
    #
    # REQUIRED before running this cell:
    #   1. Upload your local 'raw' folder to kaggle.com/datasets as a new
    #      dataset (private is fine).
    #   2. In this notebook's right sidebar: Add Input -> search for and
    #      attach that dataset.
    #   That's it -- no path or dataset-name needs to be typed here. The cell
    #   below auto-detects wherever the CSVs actually landed under
    #   /kaggle/input/, regardless of what you named the dataset or how deep
    #   Kaggle nested it.
    kaggle_input = Path('/kaggle/input')
    if not kaggle_input.exists() or not any(kaggle_input.iterdir()):
        raise FileNotFoundError(
            "/kaggle/input is empty -- attach your raw-data dataset via "
            "'Add Input' in the right sidebar before running this cell."
        )

    # Look for a known target-metric filename first (most specific, avoids
    # false positives from an unrelated dataset also being attached); fall
    # back to "any CSV at all" if that exact name isn't found near the surface.
    _candidates = sorted(
        kaggle_input.glob('**/kpi_container_cpu_usage_seconds_total.csv'),
        key=lambda p: len(p.parts)
    )
    if not _candidates:
        _candidates = sorted(kaggle_input.glob('**/*.csv'), key=lambda p: len(p.parts))

    if not _candidates:
        raise FileNotFoundError(
            f"No CSV files found anywhere under {kaggle_input}. Check that your "
            f"dataset was attached (right sidebar -> Add Input) and that it "
            f"actually contains the raw/ folder's CSVs."
        )

    # Walk up from the shallowest found CSV to the 'raw' root: this pipeline's
    # expected layout is <raw>/<complex|single>/<caseN>/<container|istio>/kpi_*.csv,
    # so the root is one level above the 'complex'/'single' split.
    raw_path = _candidates[0].parent
    while raw_path.name not in ('complex', 'single') and raw_path != raw_path.parent:
        raw_path = raw_path.parent
    if raw_path.name in ('complex', 'single'):
        raw_path = raw_path.parent

    all_csv_files = list(raw_path.glob('**/*.csv'))
    print(f"✓ Auto-detected raw folder at: {raw_path}")
    print(f"✓ Found {len(all_csv_files)} CSV files")
    if len(all_csv_files) == 0:
        print("❌ 0 CSV files found under the detected raw_path -- inspect "
              "/kaggle/input/ manually, the dataset's internal structure may "
              "not match the expected <complex|single>/<caseN>/... layout.")

else:
    print("To use this notebook, please run it in Google Colab or Kaggle")
    print("Open in Colab: https://colab.research.google.com")

✓ Auto-detected raw folder at: /kaggle/input/datasets/thanakaran/raw-data/raw
✓ Found 109 CSV files


# STEP 2: Setup Imports & Configure Paths

In [49]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from typing import List, Dict, Tuple
import json
import warnings
warnings.filterwarnings('ignore')

# Configure logging. force=True is required in Colab specifically: basicConfig()
# is a no-op if the root logger already has handlers attached (which Colab's
# kernel setup, or a prior cell execution in a long-lived session, commonly does)
# -- without force=True, every logger.info()/logger.warning() call below silently
# produces NO visible output at all, which previously hid the exact diagnostic
# line needed to explain a missing-case data problem.
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)
logger = logging.getLogger(__name__)

print("✓ All imports successful")

✓ All imports successful


In [50]:
# Setup Paths -- platform-aware: Colab has Drive mounted (persists across
# restarts), Kaggle has no Drive mount at all (everything lives on Kaggle's own
# local disk for the life of this kernel session only).
if IN_COLAB:
    gd_raw_path    = Path('/content/drive/My Drive/raw')
    output_base    = Path('/content/processed_data')
    # Sequences written DIRECTLY to Drive — too large for /content/ disk, and
    # Drive persists across Colab runtime restarts (local /content/ does not).
    sequences_path = Path('/content/drive/My Drive/processed_data/sequences')
elif IN_KAGGLE:
    gd_raw_path    = raw_path  # set by the Kaggle-Dataset auto-detection cell above
    output_base    = Path('/kaggle/working/processed_data')
    # No Drive mount on Kaggle -- everything lives on Kaggle's own local disk.
    # This does NOT persist across sessions the way Drive does in Colab -- it
    # only survives for the life of this kernel session (or until you commit
    # a notebook version, which snapshots /kaggle/working as output). Fine for
    # temporary testing; re-download/re-generate on a fresh session.
    sequences_path = Path('/kaggle/working/processed_data/sequences')
else:
    gd_raw_path    = Path('/content/drive/My Drive/raw')
    output_base    = Path('/content/processed_data')
    sequences_path = Path('/content/drive/My Drive/processed_data/sequences')

processed_path = output_base / 'processed'
merged_path    = output_base / 'merged'
sequences_path.mkdir(parents=True, exist_ok=True)

print("Path Configuration:")
print(f"  Raw input       : {gd_raw_path}")
print(f"  Merged CSVs     : {merged_path}  (local)")
print(f"  Sequences       : {sequences_path}  "
      f"({'Google Drive - persists' if IN_COLAB else 'Kaggle local disk - session only' if IN_KAGGLE else 'local'})")

log_memory("After setup")

Path Configuration:
  Raw input       : /kaggle/input/datasets/thanakaran/raw-data/raw
  Merged CSVs     : /kaggle/working/processed_data/merged  (local)
  Sequences       : /kaggle/working/processed_data/sequences  (Kaggle local disk - session only)
💾 [After setup] RAM: 8435.9 MB


8435.92578125

# STEP 1 + 2: Load Raw Data & Pivot to Wide Format

In [51]:
# Case identity is encoded in the raw CSV's FOLDER PATH, not its filename --
# confirmed via diagnostic: real layout is
#   <raw_root>/<complex|single>/<case1|case2>/<container|istio>/kpi_<metric>.csv
# e.g. 'complex/case1/container/kpi_container_cpu_usage_seconds_total.csv'.
# Filenames are just 'kpi_<metric_name>.csv' -- one file per metric per case, not
# one file containing multiple metrics as originally assumed.
VALID_CASE_NAMES = ['complex_case1', 'complex_case2', 'single_case1', 'single_case2']

def _infer_case_source(csv_path: Path) -> str:
    """Infer case identity ('complex_case1' etc.) from the folder-path components
    of csv_path, not the filename. Returns 'unknown' -- never a guess -- if the
    expected <complex|single>/<case1|case2> folder pattern isn't found."""
    parts = {p.lower() for p in csv_path.parts}
    complexity = next((p for p in ('complex', 'single') if p in parts), None)
    case_num   = next((p for p in ('case1', 'case2') if p in parts), None)
    if complexity and case_num:
        return f"{complexity}_{case_num}"
    return 'unknown'


def load_data_from_drive_optimized(raw_path: Path) -> pd.DataFrame:
    """Load CSV files and pivot from long format to wide format.

    Raw CSV format (long):
        timestamp | cmdb_id | kpi_name | value

    Output format (wide):
        timestamp | cmdb_id | new_container_id | case_source | metric_1 | metric_2 | ...

    case_source is inferred per-file from its FOLDER PATH (see _infer_case_source)
    and carried through the pivot so Phase 1's split can be done by case (roadmap
    Option A) instead of a case-blind global chronological cut.

    new_container_id is assigned from (case_source, cmdb_id) TOGETHER, not cmdb_id
    alone: the same service/replica name (e.g. 'observe.cartservice-0') recurs
    across different cases, and train combines two cases (complex_case2 +
    single_case2). Keying on cmdb_id alone would silently merge two unrelated
    cases' time series under one "container" -- the sequence generator would then
    build a 240-step window that jumps from one case's data straight into an
    unrelated case's data at the boundary. Keying on (case_source, cmdb_id) keeps
    every case's copy of a service as a genuinely separate container.

    Fails LOUDLY (raises) if any of the 4 expected cases produces zero rows in the
    final output, instead of silently continuing -- a missing case used to only
    surface several cells later as an empty train/val/test split and a confusing
    "No valid sequences produced" error during sequence generation, far from the
    real cause.
    """
    logger.info(f"Loading data from Google Drive (RAM-optimized)...")

    csv_files = sorted(raw_path.glob('**/*.csv'))
    logger.info(f"Found {len(csv_files)} CSV files")

    if not csv_files:
        logger.error("No CSV files found!")
        return None

    # Pre-flight: count raw files found per expected case BEFORE loading anything.
    # Catches a missing/empty case folder immediately with a clear message.
    file_counts_by_case = {name: 0 for name in VALID_CASE_NAMES}
    file_counts_by_case['unknown'] = 0
    for f in csv_files:
        file_counts_by_case[_infer_case_source(f)] += 1
    logger.info("Raw CSV file counts by case (before any KPI filtering):")
    for name in VALID_CASE_NAMES + ['unknown']:
        logger.info(f"    {name:16s}: {file_counts_by_case[name]:>4} files")
    _empty_case_folders = [name for name in VALID_CASE_NAMES if file_counts_by_case[name] == 0]
    if _empty_case_folders:
        logger.error(f"  No raw CSV files found at all for case(s): {_empty_case_folders} "
                      f"-- check that {raw_path}/<complexity>/<caseN>/ exists and contains .csv files.")

    # Target metrics to keep
    target_metrics = [
        'container_cpu_usage_seconds_total',
        'container_cpu_system_seconds_total',
        'container_cpu_user_seconds_total',
        'container_memory_usage_bytes',
        'container_memory_working_set_bytes',
        'container_memory_rss',
        'container_memory_cache'
    ]

    all_dfs = []
    # Diagnostic counters -- helps reconcile row counts against earlier documented
    # totals (e.g. if a prior analysis reported a different row count, this trail
    # shows exactly how many rows came from each case, pre- and post-KPI-filter).
    case_raw_rows  = {}
    case_kept_rows = {}

    for idx, csv_file in enumerate(csv_files, 1):
        try:
            case_name = _infer_case_source(csv_file)
            df = pd.read_csv(csv_file)
            raw_n = len(df)
            # Keep only rows for target metrics
            if 'kpi_name' in df.columns:
                df = df[df['kpi_name'].isin(target_metrics)]
            df['case_source'] = case_name
            kept_n = len(df)

            case_raw_rows[case_name]  = case_raw_rows.get(case_name, 0) + raw_n
            case_kept_rows[case_name] = case_kept_rows.get(case_name, 0) + kept_n

            all_dfs.append(df)
            if idx % 10 == 0:
                logger.info(f"  Loaded {idx}/{len(csv_files)} files")
        except Exception as e:
            logger.warning(f"  Error loading {csv_file.name}: {e}")

    if not all_dfs:
        logger.error("Failed to load any CSV files")
        return None

    logger.info("Row counts by case_source (raw -> after KPI filter):")
    for case_name in sorted(case_raw_rows):
        logger.info(f"    {case_name:16s}: {case_raw_rows[case_name]:>10,} -> {case_kept_rows[case_name]:>10,}")
    _zero_kept_cases = [name for name in VALID_CASE_NAMES
                         if file_counts_by_case.get(name, 0) > 0 and case_kept_rows.get(name, 0) == 0]
    if _zero_kept_cases:
        logger.error(f"  These cases have raw files but ZERO rows survive the target-metric filter: "
                      f"{_zero_kept_cases} -- their files exist but contain none of: {target_metrics}")
    if 'unknown' in case_raw_rows:
        logger.warning(f"  {case_raw_rows['unknown']:,} raw rows came from files that didn't match any "
                        f"known case folder pattern -- check VALID_CASE_NAMES above against your actual folder layout.")

    # Concatenate all long-format data
    combined = pd.concat(all_dfs, ignore_index=True)
    del all_dfs
    gc.collect()
    logger.info(f"Combined long-format rows: {len(combined):,}")

    # PIVOT: long format → wide format (one column per metric).
    #
    # case_source is included IN THE PIVOT KEY, not merged in afterward. The 4
    # cases share the same (timestamp, cmdb_id) space -- the same service name at
    # the same timestamp exists in complex_case1, complex_case2, single_case1, AND
    # single_case2, each with different metric values. Pivoting on
    # (timestamp, cmdb_id) alone would collapse all 4 cases' rows for that pair
    # into one (aggfunc='first' silently keeps whichever case happened to be
    # concatenated first -- alphabetically 'complex' before 'single' -- and
    # discards the other 3 cases' actual values, not just their case label). This
    # is exactly what caused single_case1/single_case2 to disappear entirely: they
    # collided with complex_case1/complex_case2 on every shared (timestamp,
    # cmdb_id) pair and always lost the tiebreak. Including case_source in the
    # index keeps each case's row distinct even when timestamp and cmdb_id match.
    logger.info("Pivoting long → wide format...")
    pivoted = combined.pivot_table(
        index=['timestamp', 'cmdb_id', 'case_source'],
        columns='kpi_name',
        values='value',
        aggfunc='first'
    ).reset_index()
    pivoted.columns.name = None
    del combined
    gc.collect()

    # Hard-fail check: refuse to silently continue if a whole expected case
    # produced zero rows in the final pivoted output. This used to only surface
    # much later as an empty train/val/test split and a "No valid sequences
    # produced" error during sequence generation, several cells downstream of the
    # real cause.
    _present_cases = set(pivoted['case_source'].unique())
    _missing_cases = [name for name in VALID_CASE_NAMES if name not in _present_cases]
    if _missing_cases:
        logger.error(f"FATAL: these expected cases produced ZERO rows in the final dataset: {_missing_cases}")
        logger.error(f"  Raw file counts found: {file_counts_by_case}")
        logger.error(f"  Kept-row counts after KPI filter: {case_kept_rows}")
        raise RuntimeError(
            f"Data loading produced zero rows for case(s) {_missing_cases}. "
            f"See the file-count and row-count breakdown logged above to find the cause "
            f"(missing folder, empty folder, or files with unexpected kpi_name values) "
            f"before continuing -- proceeding would silently produce empty train/val/test splits."
        )

    # Assign new_container_id from (case_source, cmdb_id) TOGETHER -- not cmdb_id
    # alone -- so the same service name in two different cases becomes two
    # distinct containers (see docstring above).
    container_key = pivoted['case_source'].astype(str) + '::' + pivoted['cmdb_id'].astype(str)
    unique_keys = sorted(container_key.unique())
    key_to_id = {k: f"container_{i+1}" for i, k in enumerate(unique_keys)}
    pivoted['new_container_id'] = container_key.map(key_to_id)
    logger.info(f"Assigned IDs to {len(unique_keys)} unique (case, container) pairs "
                f"({pivoted['cmdb_id'].nunique()} distinct cmdb_id values across "
                f"{pivoted['case_source'].nunique()} cases)")

    # Convert metrics to float32 to save RAM (case_source/new_container_id are strings, left alone)
    metric_cols = [c for c in pivoted.columns if c not in ['timestamp', 'cmdb_id', 'new_container_id', 'case_source']]
    for col in metric_cols:
        pivoted[col] = pd.to_numeric(pivoted[col], errors='coerce').astype(np.float32)

    # Sort by timestamp
    pivoted.sort_values(['timestamp', 'cmdb_id'], inplace=True)
    pivoted.reset_index(drop=True, inplace=True)

    logger.info(f"✓ Final shape: {pivoted.shape[0]:,} rows × {pivoted.shape[1]} columns")
    logger.info(f"  Columns: {list(pivoted.columns)}")
    return pivoted


df = load_data_from_drive_optimized(gd_raw_path)

if df is not None:
    print(f"✓ Data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Containers: {df['new_container_id'].nunique()}")
    print(f"  Rows by case_source: {df['case_source'].value_counts().to_dict()}")
    log_memory("After loading data")
else:
    print("❌ Failed to load data")

2026-07-24 11:43:53,904 - INFO - Loading data from Google Drive (RAM-optimized)...
2026-07-24 11:43:53,919 - INFO - Found 109 CSV files
2026-07-24 11:43:53,920 - INFO - Raw CSV file counts by case (before any KPI filtering):
2026-07-24 11:43:53,920 - INFO -     complex_case1   :   27 files
2026-07-24 11:43:53,921 - INFO -     complex_case2   :   28 files
2026-07-24 11:43:53,921 - INFO -     single_case1    :   27 files
2026-07-24 11:43:53,922 - INFO -     single_case2    :   27 files
2026-07-24 11:43:53,922 - INFO -     unknown         :    0 files
2026-07-24 11:43:55,465 - INFO -   Loaded 10/109 files
2026-07-24 11:43:58,838 - INFO -   Loaded 20/109 files
2026-07-24 11:44:01,448 - INFO -   Loaded 30/109 files
2026-07-24 11:44:02,023 - INFO -   Loaded 40/109 files
2026-07-24 11:44:03,864 - INFO -   Loaded 50/109 files
2026-07-24 11:44:04,369 - INFO -   Loaded 60/109 files
2026-07-24 11:44:04,822 - INFO -   Loaded 70/109 files
2026-07-24 11:44:06,242 - INFO -   Loaded 80/109 files
2026-

✓ Data loaded: 401,004 rows × 11 columns
  Columns: ['timestamp', 'cmdb_id', 'case_source', 'container_cpu_system_seconds_total', 'container_cpu_usage_seconds_total', 'container_cpu_user_seconds_total', 'container_memory_cache', 'container_memory_rss', 'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'new_container_id']
  Containers: 108
  Rows by case_source: {'complex_case1': 223830, 'complex_case2': 77787, 'single_case1': 60507, 'single_case2': 38880}
💾 [After loading data] RAM: 8440.0 MB


## Step 2b: Pre-Flight Integrity Check

Guards against two specific regressions that have already happened once each in
this project's history: case detection silently reverting to filename-based
matching (100% data loss), and the pivot step silently reverting to a 2-key
index that collapses 3 of 4 cases' data. Both were caused by a stale editor
buffer overwriting this notebook file outside of Colab. This check inspects the
live function source at runtime and hard-fails immediately if either regression
is present, rather than letting it silently corrupt everything downstream.

In [52]:
print("="*70)
print("PRE-FLIGHT INTEGRITY CHECK (guards against known regressions)")
print("="*70)

import inspect

_checks = []

# Guard 1: case detection must use folder-path parsing, not filename substrings
# -- V2's regression: filename-based matching silently tagged 100% of rows
# 'unknown', emptying every split.
_src_infer = inspect.getsource(_infer_case_source)
_checks.append((
    "_infer_case_source uses folder-path parsing (csv_path.parts)",
    "csv_path.parts" in _src_infer,
))

# Guard 2: pivot must key on (timestamp, cmdb_id, case_source), not
# (timestamp, cmdb_id) alone -- V1/V3's regression: a 2-key pivot silently
# collapsed 3 of 4 cases' data via aggfunc='first' collisions.
_src_load = inspect.getsource(load_data_from_drive_optimized)
_checks.append((
    "pivot_table index includes case_source (3-key, not 2-key)",
    "index=['timestamp', 'cmdb_id', 'case_source']" in _src_load,
))
_checks.append((
    "no leftover case_map groupby/merge (superseded by the 3-key pivot)",
    "case_map" not in _src_load,
))

# Guard 3: new_container_id must be case-aware -- (case_source, cmdb_id), not
# cmdb_id alone (prevents cross-case container-ID collisions in train, which
# combines two cases).
_checks.append((
    "new_container_id keyed on (case_source, cmdb_id), not cmdb_id alone",
    "container_key" in _src_load and "case_source'].astype(str)" in _src_load,
))

print(f"\n  {'CHECK':<62} RESULT")
print(f"  {'-'*62} {'-'*6}")
_n_pass = 0
for name, passed in _checks:
    print(f"  {name:<62} {'PASS' if passed else 'FAIL'}")
    _n_pass += int(passed)

print(f"\n  {_n_pass}/{len(_checks)} checks passed")
if _n_pass < len(_checks):
    raise AssertionError(
        "Pre-flight integrity check FAILED -- this notebook's Phase 1 code has "
        "regressed to a version with a previously-fixed bug (see FAIL row(s) "
        "above). This has already happened twice before, from a stale editor "
        "buffer overwriting this file outside of Colab. Re-sync cell 10 from "
        "the known-good source before proceeding -- running Phase 1 now would "
        "silently reproduce a fixed bug."
    )
print("  ✅ No known regressions detected. Safe to proceed.")

PRE-FLIGHT INTEGRITY CHECK (guards against known regressions)

  CHECK                                                          RESULT
  -------------------------------------------------------------- ------
  _infer_case_source uses folder-path parsing (csv_path.parts)   PASS
  pivot_table index includes case_source (3-key, not 2-key)      PASS
  no leftover case_map groupby/merge (superseded by the 3-key pivot) PASS
  new_container_id keyed on (case_source, cmdb_id), not cmdb_id alone PASS

  4/4 checks passed
  ✅ No known regressions detected. Safe to proceed.


# STEP 3a: Single-Case Split (complex_case1 only) + Normalize (Training Stats Only)

Revised design: train/val/test all come from **complex_case1**, split **per container
chronologically 70/15/15** with a **240-row purge gap** (embargo) at each boundary.

Why the change: the earlier cross-case split (complex_case2+single_case2 -> train,
single_case1 -> val, complex_case1 -> test) failed with a confirmed ~44x cpu_usage
scale mismatch between train and test (V6 run, Step 21 evidence) -- the GRU lost to a
persistence baseline on every horizon. The pre-implementation diagnostic on
complex_case1 confirmed its internal drift is only ~2.6x early-to-late (cpu_usage),
memory targets ~flat, no NaNs, and all 27 containers are large enough for a 3-way
split + purge gaps.

The other 3 cases (complex_case2, single_case1, single_case2) are **held out entirely**
-- reserved for the drift-detection analysis (inference-only), no longer part of
train/val/test.


In [53]:
# Single-case split (revised design) -- complex_case1 ONLY for train/val/test.
# The earlier cross-case split failed with a confirmed ~44x cpu_usage scale
# mismatch between train and test (V6 run, Step 21 evidence). New design:
# per-container chronological 70/15/15 WITHIN complex_case1, with a purge gap
# (embargo) of one full lookback window at each split boundary so no val/test
# window is temporally adjacent to train targets. The other 3 cases are held
# out for the drift-detection analysis (inference-only).
SINGLE_CASE = 'complex_case1'
TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.15   # test gets the remaining 0.15
PURGE_ROWS  = 240    # == lookback_window (Phase 1 sequence config)
SAMPLE_SECS = 15.0   # modal sampling interval (pre-implementation diagnostic, check B)

if df is not None:
    logger.info(f"Single-case split: {SINGLE_CASE} only, per-container chronological "
                f"70/15/15 with a {PURGE_ROWS}-row purge gap at each boundary...")

    if 'case_source' not in df.columns:
        raise ValueError(
            "case_source column missing -- the single-case split requires the case_source "
            "tracking added in the data-loading cell above to have run successfully."
        )

    case_df = df[df['case_source'] == SINGLE_CASE].copy()
    if case_df.empty:
        raise ValueError(
            f"No rows found for case_source == '{SINGLE_CASE}' -- check the raw data "
            f"folder structure and CASE_NAME_PATTERNS in the loading cell."
        )
    _drift_counts = df[df['case_source'] != SINGLE_CASE]['case_source'].value_counts().to_dict()
    logger.info(f"  Held out for drift analysis (in NO split): {_drift_counts}")
    _case_total = len(case_df)
    del df
    gc.collect()

    # Per-container chronological split -- every container contributes to all
    # three splits proportionally; no container ever spans a boundary window.
    case_df.sort_values(['new_container_id', 'timestamp'], inplace=True)

    _train_parts, _val_parts, _test_parts = [], [], []
    for _cid, _g in case_df.groupby('new_container_id', sort=False):
        _n = len(_g)
        _cut_val  = int(_n * TRAIN_FRAC)
        _cut_test = int(_n * (TRAIN_FRAC + VAL_FRAC))
        if _cut_test - _cut_val <= PURGE_ROWS or _n - _cut_test <= PURGE_ROWS:
            raise ValueError(
                f"Container {_cid} too short ({_n} rows) -- the {PURGE_ROWS}-row purge "
                f"gap would leave an empty val or test portion. Reduce PURGE_ROWS or "
                f"rebalance the split fractions."
            )
        _train_parts.append(_g.iloc[:_cut_val])
        # Purge gap: embargo the first PURGE_ROWS rows of val and test so no
        # window overlaps in time with the previous split's target range.
        _val_parts.append(_g.iloc[_cut_val + PURGE_ROWS:_cut_test])
        _test_parts.append(_g.iloc[_cut_test + PURGE_ROWS:])

    train_df = pd.concat(_train_parts).reset_index(drop=True)
    val_df   = pd.concat(_val_parts).reset_index(drop=True)
    test_df  = pd.concat(_test_parts).reset_index(drop=True)
    del case_df, _train_parts, _val_parts, _test_parts
    gc.collect()

    logger.info(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,} "
                f"(purged {2 * PURGE_ROWS} rows x {train_df['new_container_id'].nunique()} containers)")

    # ── Post-split integrity checks (runtime guards for the new split) ──────
    _checks = []
    for _name, _s in [('TRAIN', train_df), ('VAL', val_df), ('TEST', test_df)]:
        _checks.append((f"{_name} contains only {SINGLE_CASE}",
                        set(_s['case_source'].unique()) == {SINGLE_CASE}))
    _cont_train = set(train_df['new_container_id'].unique())
    _cont_val   = set(val_df['new_container_id'].unique())
    _cont_test  = set(test_df['new_container_id'].unique())
    _checks.append(("all containers present in all three splits",
                    _cont_train == _cont_val == _cont_test and len(_cont_train) > 0))
    _checks.append(("train fraction within 1% of target",
                    abs(len(train_df) / _case_total - TRAIN_FRAC) < 0.01))

    # Chronology + embargo verified in TIME, not just row counts: per container,
    # train must end before val begins (>= purge duration earlier), same for
    # val -> test. Irregular collection gaps only widen these margins, so >= is
    # the correct direction.
    _min_gap = PURGE_ROWS * SAMPLE_SECS
    _chrono_ok, _embargo_ok = True, True
    _t_end = train_df.groupby('new_container_id')['timestamp'].max()
    _v_beg = val_df.groupby('new_container_id')['timestamp'].min()
    _v_end = val_df.groupby('new_container_id')['timestamp'].max()
    _s_beg = test_df.groupby('new_container_id')['timestamp'].min()
    for _cid in _cont_train:
        if not (_t_end[_cid] < _v_beg[_cid] < _v_end[_cid] < _s_beg[_cid]):
            _chrono_ok = False
        if (_v_beg[_cid] - _t_end[_cid]) < _min_gap or (_s_beg[_cid] - _v_end[_cid]) < _min_gap:
            _embargo_ok = False
    _checks.append(("per-container chronology: train < val < test in time", _chrono_ok))
    _checks.append((f"purge embargo >= {_min_gap:.0f}s at both boundaries (all containers)", _embargo_ok))

    print(f"\n  {'POST-SPLIT INTEGRITY CHECK':<62} RESULT")
    print(f"  {'-'*62} {'-'*6}")
    _n_pass = 0
    for _name, _passed in _checks:
        print(f"  {_name:<62} {'PASS' if _passed else 'FAIL'}")
        _n_pass += int(_passed)
    print(f"\n  {_n_pass}/{len(_checks)} checks passed")
    if _n_pass < len(_checks):
        raise AssertionError(
            "Post-split integrity check FAILED -- the single-case split produced an "
            "invalid train/val/test layout (see FAIL rows above). Do not proceed to "
            "normalization/sequences with a broken split."
        )

    # Metric columns only (exclude ID/label columns) -- case_source is a string
    # column so it's already excluded by the dtype check below.
    metric_cols = [
        c for c in train_df.columns
        if c not in ['timestamp', 'cmdb_id', 'new_container_id', 'case_source']
        and train_df[c].dtype in [np.float32, np.float64]
    ]

    # Calculate stats from TRAINING split only — no data leakage
    logger.info("Calculating normalization stats from training split only...")
    train_stats = {}
    for col in metric_cols:
        mean = float(train_df[col].mean())
        std  = float(train_df[col].std())
        train_stats[col] = {'mean': mean, 'std': std if std > 0 else 1.0}

    # Apply training stats to ALL splits
    def normalize_with_train_stats(data: pd.DataFrame, name: str) -> None:
        """Normalize in-place using TRAINING statistics only."""
        for col, s in train_stats.items():
            if col in data.columns:
                data[col] = ((data[col] - s['mean']) / s['std']).astype(np.float32)
        logger.info(f"  {name}: normalized {len(train_stats)} columns using training stats")

    normalize_with_train_stats(train_df, 'TRAIN')
    normalize_with_train_stats(val_df,   'VAL')
    normalize_with_train_stats(test_df,  'TEST')

    train_norm = train_df
    val_norm   = val_df
    test_norm  = test_df

    # Persist normalization stats next to the sequences (not to a local ephemeral
    # working directory, which is wiped on every Colab restart and doesn't
    # survive between Kaggle sessions either) -- Phase 3 needs these to inverse
    # z-score predictions back into real units (CPU-seconds, bytes) for
    # MAE/RMSE/MAPE reporting. Without this file, Phase 3 running in a fresh
    # session has no way to recover train_stats at all.
    stats_path = sequences_path / 'normalization_stats_train.json'
    with open(stats_path, 'w') as f:
        json.dump(train_stats, f, indent=2)
    logger.info(f"  ✓ Saved normalization stats -> {stats_path}")

    print(f"✓ Single-case split ({SINGLE_CASE} only, 70/15/15 + purge) and normalized "
          f"(training stats only — no leakage)")
    print(f"  Train: {len(train_norm):,} rows | Val: {len(val_norm):,} | Test: {len(test_norm):,}")
    log_memory("After normalization")

2026-07-24 11:44:10,329 - INFO - Single-case split: complex_case1 only, per-container chronological 70/15/15 with a 240-row purge gap at each boundary...
2026-07-24 11:44:10,438 - INFO -   Held out for drift analysis (in NO split): {'complex_case2': 77787, 'single_case1': 60507, 'single_case2': 38880}
2026-07-24 11:44:10,796 - INFO - Train: 156,666 | Val: 27,098 | Test: 27,106 (purged 480 rows x 27 containers)
2026-07-24 11:44:10,834 - INFO - Calculating normalization stats from training split only...
2026-07-24 11:44:10,844 - INFO -   TRAIN: normalized 7 columns using training stats
2026-07-24 11:44:10,848 - INFO -   VAL: normalized 7 columns using training stats
2026-07-24 11:44:10,852 - INFO -   TEST: normalized 7 columns using training stats
2026-07-24 11:44:10,853 - INFO -   ✓ Saved normalization stats -> /kaggle/working/processed_data/sequences/normalization_stats_train.json



  POST-SPLIT INTEGRITY CHECK                                     RESULT
  -------------------------------------------------------------- ------
  TRAIN contains only complex_case1                              PASS
  VAL contains only complex_case1                                PASS
  TEST contains only complex_case1                               PASS
  all containers present in all three splits                     PASS
  train fraction within 1% of target                             PASS
  per-container chronology: train < val < test in time           PASS
  purge embargo >= 3600s at both boundaries (all containers)     PASS

  7/7 checks passed
✓ Single-case split (complex_case1 only, 70/15/15 + purge) and normalized (training stats only — no leakage)
  Train: 156,666 rows | Val: 27,098 | Test: 27,106
💾 [After normalization] RAM: 8440.2 MB


# STEP 3b: Feature Engineering (Lag + Rolling per Container)

> **Deviation from the original roadmap:** lag-diff and rolling-window features are
> not specified in the proposal's Phase 1 -- they were added as a reasonable ML
> practice on top of the raw metrics. Flagged here for the record, not hidden.

In [54]:
if train_norm is not None:
    logger.info("=" * 70)
    logger.info("STEP 3b: FEATURE ENGINEERING")
    logger.info("=" * 70)

    target_columns = [
        'container_cpu_usage_seconds_total',
        'container_memory_usage_bytes',
        'container_memory_working_set_bytes',
        'container_memory_rss'
    ]

    def engineer_features(data: pd.DataFrame, split_name: str) -> pd.DataFrame:
        """
        Add lag diff and rolling features grouped by container.
        Uses groupby so features never bleed across container boundaries.
        """
        logger.info(f"Engineering features for {split_name}...")

        if 'new_container_id' not in data.columns:
            logger.warning(f"  {split_name}: no new_container_id — skipping feature engineering")
            return data

        available = [c for c in target_columns if c in data.columns]
        if not available:
            logger.warning(f"  {split_name}: no target columns found")
            return data

        data = data.sort_values(['new_container_id', 'timestamp']).reset_index(drop=True)

        grp = data.groupby('new_container_id')

        for col in available:
            # Lag difference features (within each container)
            for lag in [1, 2, 3]:
                data[f'{col}_DIFF_{lag}'] = (
                    grp[col].diff(lag).fillna(0).astype(np.float32)
                )
            # Rolling mean (within each container)
            data[f'{col}_ROLLING_MEAN_3'] = (
                grp[col]
                .transform(lambda x: x.rolling(3, min_periods=1).mean())
                .astype(np.float32)
            )
            # Rolling std (within each container)
            data[f'{col}_ROLLING_STD_3'] = (
                grp[col]
                .transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0))
                .astype(np.float32)
            )

        logger.info(f"  {split_name}: {len(data.columns)} total columns")
        return data

    train_feat = engineer_features(train_norm, 'TRAIN')
    val_feat   = engineer_features(val_norm,   'VAL')
    test_feat  = engineer_features(test_norm,  'TEST')

    # Task 3.3 verification: the case-based split (Task 1.3) doesn't change which
    # columns get engineered (feature engineering is a per-container groupby,
    # independent of how rows were split into train/val/test) -- confirm that
    # explicitly instead of assuming it.
    expected_feature_count = 7 + 4 * 5  # 7 raw metrics + 4 targets x (3 lag diffs + mean + std)
    for _name, _fdf in [('TRAIN', train_feat), ('VAL', val_feat), ('TEST', test_feat)]:
        _engineered_cols = [c for c in _fdf.columns
                             if c not in ['timestamp', 'cmdb_id', 'new_container_id', 'case_source']]
        assert len(_engineered_cols) == expected_feature_count, (
            f"{_name}: expected {expected_feature_count} feature columns after engineering, "
            f"got {len(_engineered_cols)} -- check target_columns / engineer_features for a mismatch."
        )
    logger.info(f"  ✓ Verified {expected_feature_count} feature columns on all 3 splits (unaffected by case-based split)")

    # Save to disk — create merged_path only if needed
    merged_path.mkdir(parents=True, exist_ok=True)

    train_feat.to_csv(merged_path / 'train_data_with_features.csv', index=False)
    del train_feat, train_norm
    gc.collect()
    logger.info("  ✓ Train saved and cleared from RAM")

    val_feat.to_csv(merged_path / 'val_data_with_features.csv', index=False)
    del val_feat, val_norm
    gc.collect()
    logger.info("  ✓ Val saved and cleared from RAM")

    test_feat.to_csv(merged_path / 'test_data_with_features.csv', index=False)
    del test_feat, test_norm
    gc.collect()
    logger.info("  ✓ Test saved and cleared from RAM")

    print("✓ Feature engineering complete")
    print(f"  Files saved to: {merged_path}")
    log_memory("After feature engineering")

2026-07-24 11:44:10,986 - INFO - ======================================================================
2026-07-24 11:44:10,987 - INFO - STEP 3b: FEATURE ENGINEERING
2026-07-24 11:44:10,988 - INFO - ======================================================================
2026-07-24 11:44:10,989 - INFO - Engineering features for TRAIN...
2026-07-24 11:44:11,156 - INFO -   TRAIN: 31 total columns
2026-07-24 11:44:11,157 - INFO - Engineering features for VAL...
2026-07-24 11:44:11,232 - INFO -   VAL: 31 total columns
2026-07-24 11:44:11,233 - INFO - Engineering features for TEST...
2026-07-24 11:44:11,307 - INFO -   TEST: 31 total columns
2026-07-24 11:44:11,309 - INFO -   ✓ Verified 27 feature columns on all 3 splits (unaffected by case-based split)
2026-07-24 11:44:15,079 - INFO -   ✓ Train saved and cleared from RAM
2026-07-24 11:44:15,823 - INFO -   ✓ Val saved and cleared from RAM
2026-07-24 11:44:16,581 - INFO -   ✓ Test saved and cleared from RAM


✓ Feature engineering complete
  Files saved to: /kaggle/working/processed_data/merged
💾 [After feature engineering] RAM: 8441.0 MB


# STEP 4: Sliding Window Sequence Generation

In [55]:
from numpy.lib.stride_tricks import sliding_window_view

class UltraFastSequenceGeneratorOptimized:
    """
    Fully vectorized sliding-window sequence generator.

    Key design:
      - sliding_window_view  : creates all X windows at once (no Python loop over positions)
      - Shared X, multi-horizon y : X depends only on the lookback window, not on the
        prediction horizon, so it is generated ONCE per split. y stacks all horizons
        together with shape (N, n_horizons, n_targets). The previous version generated
        a full separate X copy per horizon (10x duplication -- ~69GB total on this
        dataset) even though 9 of those 10 copies were near-identical; this version
        fixes that at the source instead of patching around it downstream.
      - Per-container files  : each container saves its own temp .npy (small, fast writes)
      - open_memmap merge    : final .npy written in a single O(n) pass
      - y_indices            : pre-computed once, not inside any loop
      Speedup vs loop-based: ~100x for large datasets (vectorized window creation),
      plus ~n_horizons x less disk I/O and storage vs the old per-horizon file layout.
    """

    def __init__(self, lookback_window: int = 240, max_horizon: int = 10, batch_size: int = 2000):
        self.lookback_window = lookback_window
        self.max_horizon     = max_horizon
        self.horizons        = list(range(1, max_horizon + 1))
        self.batch_size      = batch_size   # kept for API compat, not used in vectorized path
        self.target_metrics  = [
            'container_cpu_usage_seconds_total',
            'container_memory_usage_bytes',
            'container_memory_working_set_bytes',
            'container_memory_rss'
        ]
        self.feature_cols = None

    def determine_feature_columns(self, df) -> list:
        exclude = {'timestamp', 'case_source', 'cmdb_id', 'new_container_id'}
        cols = [c for c in df.columns
                if c not in exclude
                and df[c].dtype in [np.float64, np.float32, int]]
        logger.info(f"Feature columns: {len(cols)}")
        return cols

    def process_and_save_sequences(self, csv_path: str, dataset_name: str, output_dir: str) -> bool:
        """
        Vectorized sequence generation (shared X, multi-horizon y):
          1. For each container: sliding_window_view creates all X windows in one numpy call
          2. All 10 horizons' targets are gathered in ONE vectorized fancy-index call
             (tgt_matrix), producing y of shape (n_windows, n_horizons, n_targets) --
             no per-horizon Python loop, no per-horizon X duplication
          3. NaN filtering done with boolean masks (no per-row Python checks); a window
             is kept only if X is clean AND every horizon's target is non-NaN, so X and
             y always share one consistent row count
          4. Each container writes its own small temp files
          5. Final merge uses open_memmap for O(n) disk writes
        """
        df            = None
        feature_data  = None
        container_ids = None

        try:
            logger.info(f"\nGenerating sequences for {dataset_name}...")
            df = pd.read_csv(csv_path)
            logger.info(f"  {len(df):,} rows x {len(df.columns)} columns")

            if self.feature_cols is None:
                self.feature_cols = self.determine_feature_columns(df)

            if 'new_container_id' not in df.columns or 'timestamp' not in df.columns:
                logger.error("Missing new_container_id or timestamp")
                return False

            df.sort_values(['new_container_id', 'timestamp'], inplace=True)
            df.reset_index(drop=True, inplace=True)

            output_path = Path(output_dir)
            output_path.mkdir(exist_ok=True)

            # Pre-compute y column indices once
            y_indices = [self.feature_cols.index(m)
                         for m in self.target_metrics if m in self.feature_cols]
            n_feat    = len(self.feature_cols)

            feature_data  = df[self.feature_cols].values.astype(np.float32)
            container_ids = df['new_container_id'].values

            del df
            df = None
            gc.collect()

            unique_containers = np.unique(container_ids)
            logger.info(f"Processing {len(unique_containers)} containers "
                        f"(lookback={self.lookback_window}, horizons=1-{self.max_horizon}, "
                        f"shared X / multi-horizon y)...")

            sequence_count    = 0
            container_indices = []

            for idx, container_id in enumerate(unique_containers):
                if (idx + 1) % 5 == 0:
                    logger.info(f"  Container {idx+1}/{len(unique_containers)} "
                                f"({sequence_count:,} sequences so far)")

                mask   = container_ids == container_id
                c_idx  = np.where(mask)[0]
                n_rows = len(c_idx)

                min_rows = self.lookback_window + self.max_horizon + 1
                if n_rows < min_rows:
                    continue

                cont_data   = feature_data[c_idx].astype(np.float32)  # (n_rows, n_feat)
                target_data = cont_data[:, y_indices]                 # (n_rows, n_targets) -- precomputed once

                # VECTORIZED: create all X windows at once using stride tricks
                n_windows = n_rows - self.lookback_window - self.max_horizon
                if n_windows <= 0:
                    del cont_data, target_data
                    continue

                X_windows = sliding_window_view(
                    cont_data, (self.lookback_window, n_feat)
                )[:n_windows, 0, :, :]  # view: (n_windows, lookback, n_feat)

                # VECTORIZED NaN filter on X windows
                nan_in_X   = np.isnan(X_windows).any(axis=(1, 2))   # (n_windows,)
                valid_pos  = np.where(~nan_in_X)[0]                  # positions with clean X

                if len(valid_pos) == 0:
                    del cont_data, target_data, X_windows
                    gc.collect()
                    continue

                # Copy only the valid windows (necessary to own the data)
                X_clean = np.ascontiguousarray(X_windows[valid_pos], dtype=np.float32)
                del X_windows, nan_in_X
                gc.collect()

                # VECTORIZED: gather ALL horizons' targets in one fancy-index call.
                # n_windows already reserves self.max_horizon rows at the tail of each
                # container, so every valid_pos is guaranteed in-bounds for every
                # horizon 1..max_horizon -- no per-horizon bounds check needed.
                tgt_matrix = (valid_pos[:, None]
                              + self.lookback_window
                              + np.arange(self.max_horizon)[None, :])          # (n_valid, n_horizons)
                y_stack = target_data[tgt_matrix].astype(np.float32)           # (n_valid, n_horizons, n_targets)

                # A row is kept only if it is NaN-free at EVERY horizon -- keeps X and
                # y perfectly aligned with one shared row count instead of one count
                # per horizon (which is what made the old per-horizon files diverge).
                nan_in_y = np.isnan(y_stack).any(axis=(1, 2))
                keep     = ~nan_in_y

                X_final = X_clean[keep]
                y_final = y_stack[keep]
                del cont_data, target_data, X_clean, y_stack, tgt_matrix
                gc.collect()

                if len(X_final) == 0:
                    del X_final, y_final
                    continue

                c_final = np.full(len(X_final), container_id, dtype=object)

                # Save this container's slice as a temp file -- shared across all horizons
                np.save(str(output_path / f"_tmp_X_{dataset_name}_{idx}.npy"), X_final)
                np.save(str(output_path / f"_tmp_y_{dataset_name}_{idx}.npy"), y_final)
                np.save(str(output_path / f"_tmp_c_{dataset_name}_{idx}.npy"), c_final)

                sequence_count += len(X_final)
                container_indices.append(idx)

                del X_final, y_final, c_final
                gc.collect()

            logger.info(f"Total valid sequences: {sequence_count:,}")

            if sequence_count == 0:
                logger.error(f"No valid sequences produced for {dataset_name}")
                return False

            # Merge per-container temp files into final .npy using memmap
            logger.info("Merging into final .npy files (memmap)...")
            self._merge_container_files(dataset_name, output_path, container_indices, sequence_count)

            meta = {
                'horizons':        self.horizons,
                'dataset':         dataset_name,
                'n_sequences':     sequence_count,
                'lookback_window': self.lookback_window,
                'n_features':      len(self.feature_cols),
                'feature_cols':    self.feature_cols,
                'target_metrics':  self.target_metrics,
                'X_shape':         [sequence_count, self.lookback_window, len(self.feature_cols)],
                'y_shape':         [sequence_count, self.max_horizon, len(self.target_metrics)],
                'note':            'y is shared across all horizons: y[:, h-1, :] is the target for horizon h.',
            }
            with open(output_path / f"sequences_metadata_{dataset_name}.json", 'w') as f:
                json.dump(meta, f, indent=2)
            logger.info(f"  {dataset_name}: {sequence_count:,} sequences, shared across {self.max_horizon} horizons")

            return True

        except Exception as e:
            logger.error(f"Error processing {dataset_name}: {e}")
            import traceback
            logger.error(traceback.format_exc())
            return False

        finally:
            del df, feature_data, container_ids
            gc.collect()

    def _merge_container_files(self, dataset_name, output_path, container_idx_list, n_sequences):
        """
        Merge per-container temp files into final shared .npy files.
        Uses a local scratch directory for memmap because Google Drive FUSE
        (Colab) does not support mmap for large file pre-allocation.
        Finished files are then moved to the final output_path -- on Kaggle,
        output_path is already local disk (no Drive mount at all), so this is
        just a same-disk rename, still correct and still cheap.
        """
        import shutil
        n_feat    = len(self.feature_cols)
        n_targets = len(self.target_metrics)

        # Local temp paths — mmap works on local disk (Colab /content or Kaggle
        # /kaggle/working; IN_COLAB/IN_KAGGLE are notebook-global by this point)
        _local_base = '/content' if IN_COLAB else '/kaggle/working' if IN_KAGGLE else '/content'
        local_tmp = Path(_local_base) / '_merge_tmp'
        local_tmp.mkdir(parents=True, exist_ok=True)
        X_local = local_tmp / f"sequences_X_{dataset_name}.npy"
        y_local = local_tmp / f"sequences_y_{dataset_name}.npy"

        # Final destination on Drive
        X_final = output_path / f"sequences_X_{dataset_name}.npy"
        y_final = output_path / f"sequences_y_{dataset_name}.npy"
        c_path  = output_path / f"sequences_containers_{dataset_name}.npy"

        # Memmap write on local disk (avoids Drive FUSE mmap limitation)
        X_mm = np.lib.format.open_memmap(
            str(X_local), mode='w+', dtype=np.float32,
            shape=(n_sequences, self.lookback_window, n_feat))
        y_mm = np.lib.format.open_memmap(
            str(y_local), mode='w+', dtype=np.float32,
            shape=(n_sequences, self.max_horizon, n_targets))
        c_all = []

        offset = 0
        for c_idx in container_idx_list:
            X_tmp = output_path / f"_tmp_X_{dataset_name}_{c_idx}.npy"
            y_tmp = output_path / f"_tmp_y_{dataset_name}_{c_idx}.npy"
            c_tmp = output_path / f"_tmp_c_{dataset_name}_{c_idx}.npy"

            if not X_tmp.exists():
                continue

            X_b = np.load(str(X_tmp))
            y_b = np.load(str(y_tmp))
            c_b = np.load(str(c_tmp), allow_pickle=True)
            n   = len(X_b)

            X_mm[offset:offset + n] = X_b
            y_mm[offset:offset + n] = y_b
            c_all.extend(c_b.tolist())
            offset += n

            del X_b, y_b, c_b
            X_tmp.unlink()
            y_tmp.unlink()
            c_tmp.unlink()
            gc.collect()

        # Flush and close memmap before moving
        del X_mm, y_mm
        gc.collect()

        # Move from local disk to Drive
        logger.info(f"    moving merged files to Drive...")
        shutil.move(str(X_local), str(X_final))
        shutil.move(str(y_local), str(y_final))
        np.save(str(c_path), np.array(c_all, dtype=object))
        del c_all
        gc.collect()
        logger.info(f"    merged {offset} sequences -> Drive (X shared, y covers all {self.max_horizon} horizons)")

    def diagnose(self, merged_path, sequences_path) -> None:
        merged_path    = Path(merged_path)
        sequences_path = Path(sequences_path)
        print("\n" + "="*70)
        print("DIAGNOSING SEQUENCE GENERATION")
        print("="*70)
        for name, f in [('train', merged_path / 'train_data_with_features.csv'),
                        ('val',   merged_path / 'val_data_with_features.csv'),
                        ('test',  merged_path / 'test_data_with_features.csv')]:
            print(f"  {name}: exists={f.exists()}")
            if f.exists():
                s = pd.read_csv(f, nrows=2)
                print(f"    cols={list(s.columns)[:5]} | has_id={'new_container_id' in s.columns}")
        print(f"  sequences dir exists: {sequences_path.exists()}")
        if sequences_path.exists():
            files = list(sequences_path.iterdir())
            print(f"  files: {len(files)}")
            for f in sorted(files)[:10]:
                print(f"    {f.name}")

print("✓ UltraFastSequenceGeneratorOptimized (vectorized, shared-X / multi-horizon-y) ready")

✓ UltraFastSequenceGeneratorOptimized (vectorized, shared-X / multi-horizon-y) ready


In [56]:
logger.info("\n" + "#"*70)
logger.info("STEP 4: SEQUENCE GENERATION")
logger.info("#"*70)

generator = UltraFastSequenceGeneratorOptimized(lookback_window=240, max_horizon=10, batch_size=500)

datasets = [
    ('train', merged_path / 'train_data_with_features.csv'),
    ('val',   merged_path / 'val_data_with_features.csv'),
    ('test',  merged_path / 'test_data_with_features.csv'),
]

all_success = True
for data_name, csv_file in datasets:
    logger.info(f"\nProcessing {data_name.upper()}...")
    success = generator.process_and_save_sequences(str(csv_file), data_name, str(sequences_path))
    if not success:
        logger.error(f"❌ Failed to generate sequences for {data_name}")
        all_success = False
    log_memory(f"After {data_name} sequences")

if all_success:
    print("\n✓ All sequences generated successfully")
else:
    print("\n⚠ Some datasets failed — check logs above")

2026-07-24 11:44:16,731 - INFO - 
######################################################################
2026-07-24 11:44:16,732 - INFO - STEP 4: SEQUENCE GENERATION
2026-07-24 11:44:16,733 - INFO - ######################################################################
2026-07-24 11:44:16,734 - INFO - 
Processing TRAIN...
2026-07-24 11:44:16,735 - INFO - 
Generating sequences for train...
2026-07-24 11:44:17,274 - INFO -   156,666 rows x 31 columns
2026-07-24 11:44:17,275 - INFO - Feature columns: 27
2026-07-24 11:44:17,499 - INFO - Processing 27 containers (lookback=240, horizons=1-10, shared X / multi-horizon y)...
2026-07-24 11:44:19,738 - INFO -   Container 5/27 (22,220 sequences so far)
2026-07-24 11:44:22,673 - INFO -   Container 10/27 (49,989 sequences so far)
2026-07-24 11:44:25,560 - INFO -   Container 15/27 (77,740 sequences so far)
2026-07-24 11:44:28,411 - INFO -   Container 20/27 (105,505 sequences so far)
2026-07-24 11:44:31,168 - INFO -   Container 25/27 (133,257 sequenc

💾 [After train sequences] RAM: 8443.7 MB


2026-07-24 11:44:48,125 - INFO - Processing 27 containers (lookback=240, horizons=1-10, shared X / multi-horizon y)...
2026-07-24 11:44:49,657 - INFO -   Container 5/27 (3,017 sequences so far)
2026-07-24 11:44:51,616 - INFO -   Container 10/27 (6,787 sequences so far)
2026-07-24 11:44:53,540 - INFO -   Container 15/27 (10,552 sequences so far)
2026-07-24 11:44:55,394 - INFO -   Container 20/27 (14,320 sequences so far)
2026-07-24 11:44:57,281 - INFO -   Container 25/27 (18,086 sequences so far)
2026-07-24 11:44:58,463 - INFO - Total valid sequences: 20,348
2026-07-24 11:44:58,464 - INFO - Merging into final .npy files (memmap)...
2026-07-24 11:45:02,795 - INFO -     moving merged files to Drive...
2026-07-24 11:45:03,215 - INFO -     merged 20348 sequences -> Drive (X shared, y covers all 10 horizons)
2026-07-24 11:45:03,216 - INFO -   val: 20,348 sequences, shared across 10 horizons
2026-07-24 11:45:03,429 - INFO - 
Processing TEST...
2026-07-24 11:45:03,429 - INFO - 
Generating sequ

💾 [After val sequences] RAM: 8442.7 MB


2026-07-24 11:45:03,678 - INFO - Processing 27 containers (lookback=240, horizons=1-10, shared X / multi-horizon y)...
2026-07-24 11:45:05,171 - INFO -   Container 5/27 (3,018 sequences so far)
2026-07-24 11:45:07,144 - INFO -   Container 10/27 (6,789 sequences so far)
2026-07-24 11:45:08,961 - INFO -   Container 15/27 (10,557 sequences so far)
2026-07-24 11:45:10,732 - INFO -   Container 20/27 (14,327 sequences so far)
2026-07-24 11:45:12,520 - INFO -   Container 25/27 (18,094 sequences so far)
2026-07-24 11:45:13,605 - INFO - Total valid sequences: 20,356
2026-07-24 11:45:13,606 - INFO - Merging into final .npy files (memmap)...
2026-07-24 11:45:17,765 - INFO -     moving merged files to Drive...
2026-07-24 11:45:18,183 - INFO -     merged 20356 sequences -> Drive (X shared, y covers all 10 horizons)
2026-07-24 11:45:18,184 - INFO -   test: 20,356 sequences, shared across 10 horizons


💾 [After test sequences] RAM: 8442.7 MB

✓ All sequences generated successfully


# STEP 7: Verify Sequences

In [57]:
print("="*70)
print("VERIFYING SEQUENCES")
print("="*70)

try:
    X_train = np.load(sequences_path / 'sequences_X_train.npy')
    y_train = np.load(sequences_path / 'sequences_y_train.npy')

    print(f"\nTraining Data (all horizons share one X file):")
    print(f"  X shape: {X_train.shape}   (n_samples, 240, n_features)")
    print(f"  y shape: {y_train.shape}   (n_samples, 10 horizons, 4 targets) -- y[:, h-1, :] is horizon h's target")

    npy_files = list(sequences_path.glob('*.npy'))
    json_files = list(sequences_path.glob('*.json'))

    print(f"\n  Total .npy files: {len(npy_files)}   (expect 3 splits x 3 arrays = 9, not 90 -- X/y are no longer duplicated per horizon)")
    print(f"  Total .json files: {len(json_files)}  (expect 3, one per split)")

    print("\n" + "="*70)
    print("✅ SEQUENCES GENERATED SUCCESSFULLY!")
    print("="*70)

    final_mem = log_memory("Final")
    print(f"\n💾 Memory Summary:")
    print(f"  Initial: {initial_mem:.1f} MB")
    print(f"  Final:   {final_mem:.1f} MB")
    print(f"  Peak:    ~{final_mem:.1f} MB")

except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("Sequences were not generated. Running diagnostics...\n")
    generator.diagnose(merged_path, sequences_path)

VERIFYING SEQUENCES

Training Data (all horizons share one X file):
  X shape: (149916, 240, 27)   (n_samples, 240, n_features)
  y shape: (149916, 10, 4)   (n_samples, 10 horizons, 4 targets) -- y[:, h-1, :] is horizon h's target

  Total .npy files: 9   (expect 3 splits x 3 arrays = 9, not 90 -- X/y are no longer duplicated per horizon)
  Total .json files: 4  (expect 3, one per split)

✅ SEQUENCES GENERATED SUCCESSFULLY!
💾 [Final] RAM: 9595.1 MB

💾 Memory Summary:
  Initial: 8435.9 MB
  Final:   9595.1 MB
  Peak:    ~9595.1 MB


---

# Phase 2 — GRU Model Architecture & Data Pipeline

> **Prerequisites:** Run all Phase 1 cells above first.
> `sequences_path` is already defined and the `.npy` files are generated.

---

### What Phase 2 builds

| Component | Description |
|---|---|
| `SequenceDataset` | Custom Dataset wrapping (X, y) NumPy arrays |
| `load_dataset()` | Loads .npy files for a given horizon + split |
| `create_dataloader()` | Configures a DataLoader from arrays |
| `build_dataloaders()` | Builds train / val / test loaders in one call |
| `initialize_weights()` | Orthogonal (GRU) + Xavier (Linear) init |
| `GRUModel` | 2-layer GRU -> Dropout -> FC -> ReLU -> Output (batch, 4) |

## Step 2: Imports

In [58]:
# Phase 2 additional imports (PyTorch — not needed in Phase 1)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Dict, Tuple

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"sequences_path  : {sequences_path}  (exists: {sequences_path.exists()})")

PyTorch version : 2.10.0+cu128
Device          : cuda
CUDA available  : True
sequences_path  : /kaggle/working/processed_data/sequences  (exists: True)


## Step 3: SequenceDataset

A custom `Dataset` that:
- Accepts raw NumPy arrays `(X, y)` from the Phase 1 `.npy` files
- Converts them to `float32` tensors **once** in `__init__` (not per-sample)
- Exposes `__len__` and `__getitem__` so PyTorch's DataLoader can index into it

In [59]:
class SequenceDataset(Dataset):
    """
    PyTorch Dataset for pre-computed sliding-window sequences.

    Parameters
    ----------
    X : np.ndarray  shape (n_samples, seq_len, n_features)
        Input windows produced by the Phase 1 sequence generator.
        Example: (174220, 240, 27)
    y : np.ndarray  shape (n_samples, n_targets)
        Regression targets at the chosen prediction horizon.
        Example: (174220, 4)

    Notes
    -----
    Arrays are cast to float32 ONCE in __init__.
    No normalisation is applied here — sequences are already
    z-score normalised from Phase 1.
    """

    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        if len(X) != len(y):
            raise ValueError(
                f"X and y must have the same number of samples "
                f"(got X={len(X)}, y={len(y)})"
            )
        # np.ascontiguousarray ensures tensors are in contiguous memory
        # which is required for efficient GPU transfer via pin_memory
        self.X = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.ascontiguousarray(y, dtype=np.float32))

    def __len__(self) -> int:
        """Total number of samples in the dataset."""
        return len(self.X)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Return one (input_window, target) pair.

        Returns
        -------
        x : torch.Tensor  shape (seq_len, n_features)  e.g. (240, 27)
        y : torch.Tensor  shape (n_targets,)            e.g. (4,)
        """
        return self.X[idx], self.y[idx]

    # ── Informational properties ──────────────────────────────────────
    @property
    def n_samples(self)  -> int: return len(self.X)
    @property
    def seq_len(self)    -> int: return self.X.shape[1]
    @property
    def n_features(self) -> int: return self.X.shape[2]
    @property
    def n_targets(self)  -> int: return self.y.shape[1]

    def __repr__(self) -> str:
        return (f"SequenceDataset("
                f"n_samples={self.n_samples}, "
                f"seq_len={self.seq_len}, "
                f"n_features={self.n_features}, "
                f"n_targets={self.n_targets})")

print("SequenceDataset defined.")

SequenceDataset defined.


## Step 4: load_dataset()

Loads the shared `(X, y)` pair of `.npy` files for a given **split**, then slices out
the target for one **horizon**. X is generated once per split (not once per horizon --
see Phase 1's sequence generator) since the 240-step lookback window doesn't depend on
which horizon is being predicted; only y varies by horizon, and it's stored as one
`(n_samples, 10, 4)` array per split so all 10 horizons can share it.

File naming convention (from Phase 1):
```
sequences_X_{split}.npy      # shape (n_samples, 240, n_features) -- shared across horizons
sequences_y_{split}.npy      # shape (n_samples, 10, 4) -- all horizons; sliced by load_dataset()
```

In [60]:
def load_dataset(
    sequences_dir: str,
    horizon: int,
    split: str,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load the pre-generated (X, y) NumPy files for one data split, and slice out
    the target for one horizon.

    X is SHARED across all 10 horizons (it depends only on the 240-step lookback
    window, not on the prediction horizon) -- Phase 1 writes it once per split, not
    once per horizon. y is written once per split too, with shape (n_samples, 10, 4);
    this function slices out column `horizon - 1` to return the (n_samples, 4) target
    array the rest of Phase 2/3 expects.

    Parameters
    ----------
    sequences_dir : str
        Path to the directory containing .npy files.
        In Colab: '/content/drive/My Drive/processed_data/sequences'
        In Kaggle: '/kaggle/working/processed_data/sequences'
    horizon : int
        Prediction horizon, 1 to 10.
        horizon=1  -> predict 1 step ahead  (15 seconds)
        horizon=10 -> predict 10 steps ahead (2.5 minutes)
    split : str
        One of 'train', 'val', or 'test'.

    Returns
    -------
    X : np.ndarray  shape (n_samples, seq_len, n_features)
    y : np.ndarray  shape (n_samples, n_targets)  -- this horizon's targets only

    Raises
    ------
    ValueError        if horizon not in [1,10] or split is unrecognised
    FileNotFoundError if .npy files are missing from sequences_dir
    """
    valid_splits = {'train', 'val', 'test'}
    if split not in valid_splits:
        raise ValueError(f"split must be one of {valid_splits}, got '{split}'")
    if not (1 <= horizon <= 10):
        raise ValueError(f"horizon must be 1-10, got {horizon}")

    base   = Path(sequences_dir)
    X_path = base / f"sequences_X_{split}.npy"
    y_path = base / f"sequences_y_{split}.npy"

    if not X_path.exists():
        raise FileNotFoundError(f"X file not found: {X_path}")
    if not y_path.exists():
        raise FileNotFoundError(f"y file not found: {y_path}")

    X     = np.load(str(X_path))
    y_all = np.load(str(y_path))          # (n_samples, 10, 4) -- all horizons together
    y     = y_all[:, horizon - 1, :]       # this horizon's (n_samples, 4) target slice

    logger.info(f"[load_dataset] horizon={horizon} {split} | X={X.shape} y={y.shape} (sliced from y_all={y_all.shape})")
    return X, y

print("load_dataset() defined.")

load_dataset() defined.


## Step 5: create_dataloader()

Wraps `(X, y)` NumPy arrays in a PyTorch `DataLoader` with configurable settings.

Key choices:
- `shuffle=True` for **training only** — prevents the model from memorising batch order
- `num_workers=0` — safest setting in Colab (multiprocessing is restricted in notebooks)
- `pin_memory=True` when a GPU is available — speeds up CPU to GPU tensor transfer

In [61]:
def create_dataloader(
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int  = 64,
    shuffle: bool    = True,
    num_workers: int = 0,
    pin_memory: bool = False,
    drop_last: bool  = False,
) -> DataLoader:
    """
    Build a DataLoader from NumPy arrays.

    Parameters
    ----------
    X, y        : NumPy arrays (see load_dataset for shapes)
    batch_size  : Samples per gradient update. Typical: 32, 64, 128.
    shuffle     : True for train, False for val/test.
    num_workers : Parallel loading workers. Use 0 in Colab.
    pin_memory  : Set True when training on GPU (faster CPU->GPU copy).
    drop_last   : Discard final incomplete batch. Useful with BatchNorm layers.

    Returns
    -------
    DataLoader that yields (X_batch, y_batch) tensors each iteration.
    """
    dataset = SequenceDataset(X, y)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=drop_last,
    )

    logger.info(
        f"[create_dataloader] {len(dataset):,} samples | "
        f"{len(loader)} batches | batch_size={batch_size} | shuffle={shuffle}"
    )
    return loader

print("create_dataloader() defined.")

create_dataloader() defined.


## Step 6: build_dataloaders()

Convenience function — builds all three DataLoaders (train / val / test) for a
given horizon in one call with the correct shuffle settings applied automatically.

In [62]:
def build_dataloaders(
    sequences_dir: str,
    horizon: int,
    batch_size: int  = 64,
    num_workers: int = 0,
    pin_memory: bool = False,
) -> Dict[str, DataLoader]:
    """
    Load all three splits and return a dict of DataLoaders.

    Shuffle policy applied automatically:
        train -> shuffle=True
        val   -> shuffle=False
        test  -> shuffle=False

    Parameters
    ----------
    sequences_dir : str   Path to .npy files directory.
    horizon       : int   Prediction horizon (1-10).
    batch_size    : int   Shared batch size for all loaders.
    num_workers   : int   Parallel workers (0 for Colab).
    pin_memory    : bool  Pin tensors to CUDA memory when using GPU.

    Returns
    -------
    dict  {'train': DataLoader, 'val': DataLoader, 'test': DataLoader}

    Example
    -------
    loaders = build_dataloaders(str(sequences_path), horizon=1, batch_size=64)
    X_batch, y_batch = next(iter(loaders['train']))
    print(X_batch.shape)  # (64, 240, 27)
    print(y_batch.shape)  # (64, 4)
    """
    split_cfg = [
        ('train', True),
        ('val',   False),
        ('test',  False),
    ]

    loaders: Dict[str, DataLoader] = {}
    for split, shuffle in split_cfg:
        X, y = load_dataset(sequences_dir, horizon, split)
        loaders[split] = create_dataloader(
            X, y,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

    return loaders

print("build_dataloaders() defined.")

build_dataloaders() defined.


## Step 7: initialize_weights()

Called via `model.apply(initialize_weights)` which visits **every sub-module**
in the model tree one at a time.

| Layer | Weight type | Method | Why |
|---|---|---|---|
| GRU | `weight_ih` (input to hidden) | Xavier uniform | Stable activation variance at input gate |
| GRU | `weight_hh` (hidden to hidden) | **Orthogonal** | Eigenvalues = 1, prevents gradient vanish/explode through time |
| GRU | biases | Zeros | Clean start |
| Linear | weights | Xavier uniform | Stable variance across FC layers |
| Linear | biases | Zeros | Clean start |

In [63]:
def initialize_weights(module: nn.Module) -> None:
    """
    Best-practice weight initialisation for GRU + Linear layers.

    Called via:  model.apply(initialize_weights)
    PyTorch walks every sub-module and passes it here individually.

    Rules
    -----
    GRU weight_ih  -> Xavier uniform
        Keeps the variance of activations stable at the input gate.

    GRU weight_hh  -> Orthogonal initialisation
        Orthogonal matrices have unit-magnitude eigenvalues.
        Gradients neither explode nor vanish as they flow backward
        through the recurrent connections.
        This is the standard best practice for RNN weight init.

    GRU / Linear biases -> Zeros
        Neutral starting point; the model learns offsets from data.

    Linear weights -> Xavier uniform
        Keeps variance stable across the fully connected head layers.
    """
    if isinstance(module, nn.GRU):
        for name, param in module.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                nn.init.zeros_(param.data)

    elif isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

print("initialize_weights() defined.")

initialize_weights() defined.


## Step 8: GRUModel

```
Input  (batch, 240, 27)
    |
    v
GRU  (2 stacked layers, hidden=128, inter-layer dropout=0.2)
    |   only h_n[-1] is used — the top layer's final hidden state
    v   shape: (batch, 128)
Dropout (0.2)
    |
    v
Linear  128 -> 64
    |
    v
ReLU
    |
    v
Linear  64 -> 4
    |
    v
Output (batch, 4)
```

**Why `h_n[-1]` and not `gru_out[:, -1, :]`?**
Both are numerically identical. `h_n[-1]` is the explicit final hidden state
of the top GRU layer — the compressed temporal summary of the full 240-step window.

**Why no activation on the output layer?**
This is a regression task. Sigmoid/tanh would clip predictions.
MSELoss (Phase 3) expects raw unbounded values.

In [64]:
class GRUModel(nn.Module):
    """
    Stacked GRU network for multi-step container resource forecasting.

    One independent instance is trained per prediction horizon.
    horizon=1  -> model predicts 1 timestep  (15 sec) ahead
    horizon=10 -> model predicts 10 timesteps (2.5 min) ahead

    Parameters
    ----------
    input_size  : int   Features per timestep. Default 27.
    hidden_size : int   GRU hidden units per layer. Default 128.
    num_layers  : int   Stacked GRU layers. Default 2.
    output_size : int   Regression targets. Default 4.
    dropout     : float Dropout probability applied between GRU layers
                        and before the FC head. Range [0, 1).
    """

    def __init__(
        self,
        input_size:  int   = 27,
        hidden_size: int   = 128,
        num_layers:  int   = 2,
        output_size: int   = 4,
        dropout:     float = 0.2,
    ) -> None:
        super().__init__()

        self.input_size  = input_size
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.output_size = output_size
        self.dropout_p   = dropout

        # ── GRU stack ────────────────────────────────────────────────────
        # batch_first=True  expects (batch, seq_len, features)
        #                   returns (batch, seq_len, hidden_size)
        # dropout is applied between stacked layers only
        # (PyTorch ignores dropout for single-layer GRUs)
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # ── Dropout before FC head ────────────────────────────────────────
        # Applied to the final hidden state.
        # Prevents co-adaptation of hidden units; reduces overfitting.
        self.dropout = nn.Dropout(p=dropout)

        # ── Fully connected head ──────────────────────────────────────────
        # Two-layer MLP compresses hidden_size -> 64 -> output_size.
        # The intermediate 64-unit layer learns non-linear combinations
        # of the GRU's compressed temporal representation.
        self.fc1  = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(64, output_size)

        # Apply weight initialisation to every sub-module in the model
        self.apply(initialize_weights)

        logger.info(
            f"[GRUModel] input={input_size} hidden={hidden_size} "
            f"layers={num_layers} output={output_size} dropout={dropout} | "
            f"params={self._count_params():,}"
        )

    # ── Forward pass ─────────────────────────────────────────────────────

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : torch.Tensor  shape (batch_size, seq_len, input_size)
            A batch of input windows from the DataLoader.

        Returns
        -------
        torch.Tensor  shape (batch_size, output_size)
            Raw regression predictions (no output activation).

        Flow
        ----
        1. GRU processes the full 240-timestep sequence.
           gru_out : (batch, 240, hidden_size) -- all timestep outputs (unused)
           h_n     : (num_layers, batch, hidden_size) -- final hidden states

        2. h_n[-1] extracts the TOP layer's final hidden state: (batch, hidden_size)
           This is the most abstract temporal summary of the input window.

        3. Dropout -> FC1 -> ReLU -> FC2 maps it to 4 target predictions.
        """
        # Step 1: run GRU over the full sequence
        gru_out, h_n = self.gru(x)

        # Step 2: take only the top layer's final hidden state
        last_hidden = h_n[-1]            # (batch, hidden_size)

        # Step 3: FC head with dropout
        out = self.dropout(last_hidden)  # (batch, hidden_size)
        out = self.fc1(out)              # (batch, 64)
        out = self.relu(out)             # (batch, 64)
        out = self.fc2(out)              # (batch, output_size)

        return out

    # ── Utility methods ───────────────────────────────────────────────────

    def _count_params(self) -> int:
        """Return total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def summary(self) -> None:
        """Print a human-readable architecture summary."""
        sep = "=" * 58
        print(sep)
        print("GRUModel - Architecture Summary")
        print(sep)
        print(f"  Input size   : {self.input_size}  (features per timestep)")
        print(f"  Hidden size  : {self.hidden_size}")
        print(f"  GRU layers   : {self.num_layers}")
        print(f"  Dropout      : {self.dropout_p}")
        print(f"  Output size  : {self.output_size}  (regression targets)")
        print("-" * 58)
        print(f"  Layer           Output shape")
        print(f"  GRU (stack)     (batch, 240, {self.hidden_size}) -> h_n[-1]: (batch, {self.hidden_size})")
        print(f"  Dropout         (batch, {self.hidden_size})")
        print(f"  Linear  fc1     (batch, 64)")
        print(f"  ReLU            (batch, 64)")
        print(f"  Linear  fc2     (batch, {self.output_size})")
        print(sep)
        print(f"  Trainable parameters : {self._count_params():,}")
        print(sep)

    def __repr__(self) -> str:
        return (f"GRUModel(input={self.input_size}, hidden={self.hidden_size}, "
                f"layers={self.num_layers}, output={self.output_size}, "
                f"dropout={self.dropout_p}, params={self._count_params():,})")

print("GRUModel defined.")

GRUModel defined.


## Step 9: Smoke Test

Verifies every component works correctly before connecting to real data.

**Checks:**
1. `GRUModel` forward pass produces correct output shape `(64, 4)`
2. `SequenceDataset` wraps arrays correctly and is indexable
3. `DataLoader` yields batches with the right shapes
4. `build_dataloaders()` loads real `.npy` files from the sequences directory

In [65]:
print("=" * 58)
print("SMOKE TEST - Phase 2 Components")
print("=" * 58)

# ── Test 1: Model forward pass ────────────────────────────────
print("""
[1] GRUModel forward pass""")
model = GRUModel(input_size=27, hidden_size=128, num_layers=2,
                 output_size=4, dropout=0.2)
model.summary()
dummy_input = torch.randn(64, 240, 27)
dummy_out   = model(dummy_input)
print(f"    Input  shape : {tuple(dummy_input.shape)}")
print(f"    Output shape : {tuple(dummy_out.shape)}")
assert dummy_out.shape == (64, 4)
print("    PASSED")

# ── Test 2: SequenceDataset ───────────────────────────────────
print("""
[2] SequenceDataset""")
X_fake = np.random.randn(500, 240, 27).astype(np.float32)
y_fake = np.random.randn(500, 4).astype(np.float32)
ds = SequenceDataset(X_fake, y_fake)
print(f"    {ds}")
x0, y0 = ds[0]
assert x0.shape == (240, 27)
assert y0.shape == (4,)
print("    PASSED")

# ── Test 3: DataLoader ────────────────────────────────────────
print("""
[3] create_dataloader""")
loader = create_dataloader(X_fake, y_fake, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))
print(f"    Batch X : {tuple(xb.shape)}")
print(f"    Batch y : {tuple(yb.shape)}")
assert xb.shape == (64, 240, 27)
assert yb.shape == (64, 4)
print("    PASSED")

# ── Test 4: Real .npy files (mmap -- no full file load, shared X / multi-horizon y) ───
print("""
[4] Real .npy files (horizon=1, shared X / multi-horizon y)""")
import os

if not sequences_path.exists():
    print(f"    SKIPPED - path not found: {sequences_path}")
else:
    # Step 1: Check file sizes (fast — no data read)
    print("    File size check:")
    all_ok = True
    for split in ['train', 'val', 'test']:
        X_f = sequences_path / f"sequences_X_{split}.npy"
        y_f = sequences_path / f"sequences_y_{split}.npy"
        x_mb = os.path.getsize(X_f) // (1024*1024) if X_f.exists() else 0
        y_mb = os.path.getsize(y_f) // (1024*1024) if y_f.exists() else 0
        ok   = x_mb > 0 and y_mb > 0
        print(f"      {split:5s}  X={x_mb:,}MB  y={y_mb:,}MB  [{'OK' if ok else 'EMPTY/MISSING'}]")
        if not ok:
            all_ok = False

    if not all_ok:
        print("    Some files are empty or missing. Re-run Phase 1.")
    else:
        # Step 2: Load with mmap_mode='r' — reads only one batch, not the full file
        print("    Loading one batch per split using mmap, slicing horizon 1 out of y (fast)...")
        passed = True
        for split in ['train', 'val', 'test']:
            X_f = sequences_path / f"sequences_X_{split}.npy"
            y_f = sequences_path / f"sequences_y_{split}.npy"
            try:
                X_mm = np.load(str(X_f), mmap_mode='r')   # memory-mapped, no full load
                y_mm = np.load(str(y_f), mmap_mode='r')   # (n, 10, 4) -- all horizons
                n    = len(X_mm)
                # Read only first 64 rows (one batch), horizon 1 slice of y
                X_batch = torch.from_numpy(np.array(X_mm[:64], dtype=np.float32))
                y_batch = torch.from_numpy(np.array(y_mm[:64, 0, :], dtype=np.float32))
                n_batches = int(np.ceil(n / 64))
                print(f"      {split:5s}  n={n:,}  X={tuple(X_batch.shape)}  y_h1={tuple(y_batch.shape)}  "
                      f"y_all_shape={y_mm.shape}  batches={n_batches}")
                assert X_batch.shape == (64, 240, 27), f"Wrong X shape: {X_batch.shape}"
                assert y_batch.shape == (64, 4),       f"Wrong y shape: {y_batch.shape}"
                assert y_mm.shape[1] == 10, f"Expected y to cover 10 horizons, got shape {y_mm.shape}"
                del X_mm, y_mm, X_batch, y_batch
            except Exception as e:
                print(f"      {split:5s}  FAILED: {e}")
                passed = False
        print("    PASSED" if passed else "    FAILED — re-run Phase 1")

print("""
""" + "=" * 58)
print("ALL SMOKE TESTS PASSED")
print("=" * 58)
print("""
Phase 2 complete. Waiting for approval to proceed to Phase 3.""")

2026-07-24 11:45:23,285 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876


SMOKE TEST - Phase 2 Components

[1] GRUModel forward pass
GRUModel - Architecture Summary
  Input size   : 27  (features per timestep)
  Hidden size  : 128
  GRU layers   : 2
  Dropout      : 0.2
  Output size  : 4  (regression targets)
----------------------------------------------------------
  Layer           Output shape
  GRU (stack)     (batch, 240, 128) -> h_n[-1]: (batch, 128)
  Dropout         (batch, 128)
  Linear  fc1     (batch, 64)
  ReLU            (batch, 64)
  Linear  fc2     (batch, 4)
  Trainable parameters : 167,876
    Input  shape : (64, 240, 27)
    Output shape : (64, 4)
    PASSED

[2] SequenceDataset


2026-07-24 11:45:23,506 - INFO - [create_dataloader] 500 samples | 8 batches | batch_size=64 | shuffle=True


    SequenceDataset(n_samples=500, seq_len=240, n_features=27, n_targets=4)
    PASSED

[3] create_dataloader
    Batch X : (64, 240, 27)
    Batch y : (64, 4)
    PASSED

[4] Real .npy files (horizon=1, shared X / multi-horizon y)
    File size check:
      train  X=3,705MB  y=22MB  [OK]
      val    X=502MB  y=3MB  [OK]
      test   X=503MB  y=3MB  [OK]
    Loading one batch per split using mmap, slicing horizon 1 out of y (fast)...
      train  n=149,916  X=(64, 240, 27)  y_h1=(64, 4)  y_all_shape=(149916, 10, 4)  batches=2343
      val    n=20,348  X=(64, 240, 27)  y_h1=(64, 4)  y_all_shape=(20348, 10, 4)  batches=318
      test   n=20,356  X=(64, 240, 27)  y_h1=(64, 4)  y_all_shape=(20356, 10, 4)  batches=319
    PASSED

ALL SMOKE TESTS PASSED

Phase 2 complete. Waiting for approval to proceed to Phase 3.


---

# Phase 3 — Training Pipeline

> **Prerequisites:** Phase 1 and Phase 2 cells must be run first.
> `sequences_path`, `device`, `GRUModel`, `build_dataloaders` must all be defined.

---

### What Phase 3 builds

| Component | Description |
|---|---|
| `TrainingConfig` | Single dataclass holding all hyperparameters |
| `MmapSequenceDataset` | Memory-mapped Dataset — loads large .npy files without OOM |
| `EarlyStopping` | Stops training when val loss stops improving |
| `train_one_epoch()` | One full pass over the training DataLoader |
| `validate()` | Evaluation pass — no gradients, no weight updates |
| `save_checkpoint()` | Saves best model weights to Google Drive |
| `train_model()` | Full loop: epochs + early stopping + LR scheduler + checkpointing |
| `train_all_horizons()` | Trains one independent GRU per horizon (1 to 10) |

## Step 10: Phase 3 Imports

In [66]:
import time
import json as json_lib
import random
from dataclasses import dataclass, field
from typing import List, Optional
from pathlib import Path

# Confirm Phase 2 components are available
assert 'GRUModel' in dir(), "Run Phase 2 cells first — GRUModel not defined"
assert 'sequences_path' in dir(), "Run Phase 1 cell 8 first — sequences_path not defined"


def set_seed(seed: int = 42) -> None:
    """Fix all random seeds for reproducibility. Called once PER HORIZON (with a
    different seed each time) inside train_all_horizons(), not just once globally --
    that way re-running a single horizon in isolation reproduces the same weights
    it had inside a full 10-horizon run, instead of depending on the RNG state left
    behind by every horizon that happened to run before it."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# Target metrics — order MUST match Phase 1's UltraFastSequenceGeneratorOptimized
# .target_metrics, since y's last dimension is ordered the same way.
TARGET_NAMES = ['cpu_usage', 'mem_usage', 'mem_working_set', 'mem_rss']
TARGET_COLUMNS = [
    'container_cpu_usage_seconds_total',
    'container_memory_usage_bytes',
    'container_memory_working_set_bytes',
    'container_memory_rss',
]

# Load Phase 1's normalization stats (mean/std per column, train-split-only) so
# Phase 3 can inverse z-score predictions back into real units for MAE/RMSE/MAPE.
# This is a hard requirement, not optional: after any Colab runtime restart, the
# in-memory `train_stats` dict from Phase 1 is gone -- this file is the only
# surviving copy, which is exactly why Phase 1's split+normalize cell now saves it.
_stats_path = sequences_path / 'normalization_stats_train.json'
assert _stats_path.exists(), (
    f"normalization_stats_train.json not found at {_stats_path} -- "
    f"re-run Phase 1's split+normalize cell (it now saves this file)."
)
with open(_stats_path) as _f:
    train_stats = json_lib.load(_f)

TARGET_MEAN = np.array([train_stats[c]['mean'] for c in TARGET_COLUMNS], dtype=np.float32)
TARGET_STD  = np.array([train_stats[c]['std']  for c in TARGET_COLUMNS], dtype=np.float32)

print(f"Device          : {device}")
print(f"sequences_path  : {sequences_path}")
print(f"Target metrics  : {TARGET_NAMES}")
print(f"Loaded normalization stats for {len(train_stats)} columns from {_stats_path.name}")
print(f"Phase 3 imports : OK")

Device          : cuda
sequences_path  : /kaggle/working/processed_data/sequences
Target metrics  : ['cpu_usage', 'mem_usage', 'mem_working_set', 'mem_rss']
Loaded normalization stats for 7 columns from normalization_stats_train.json
Phase 3 imports : OK


## Step 11: TrainingConfig

All hyperparameters live in one dataclass.
Change values here — nothing else needs editing.

| Parameter | Default | Meaning |
|---|---|---|
| `hidden_size` | 128 | GRU hidden units per layer |
| `num_layers` | 2 | Stacked GRU layers |
| `dropout` | 0.2 | Dropout probability |
| `batch_size` | 64 | Samples per gradient update |
| `learning_rate` | 1e-3 | Initial Adam learning rate |
| `num_epochs` | 50 | Maximum training epochs |
| `patience` | 10 | Early stopping patience |
| `scheduler_patience` | 5 | Epochs before LR reduction |
| `scheduler_factor` | 0.5 | LR multiplied by this on plateau |
| `horizons` | 1-10 | Which horizons to train |

In [67]:
@dataclass
class TrainingConfig:
    """
    Central configuration for the GRU training pipeline.

    All paths, model hyperparameters, and training settings
    are defined here. Passed to every training function so
    there are no magic numbers scattered through the code.
    """

    # ── Model architecture ─────────────────────────────────────────────
    input_size:  int   = 27     # features per timestep (fixed by Phase 1)
    hidden_size: int   = 128    # GRU hidden units
    num_layers:  int   = 2      # stacked GRU layers
    output_size: int   = 4      # regression targets (fixed by Phase 1)
    dropout:     float = 0.2    # dropout probability

    # ── Training ───────────────────────────────────────────────────────
    batch_size:         int   = 64      # samples per batch
    learning_rate:      float = 1e-3    # initial Adam LR
    num_epochs:         int   = 50      # max epochs per horizon
    patience:           int   = 10      # early stopping patience
    grad_clip:          float = 1.0     # gradient clipping max norm

    # ── LR scheduler (ReduceLROnPlateau) ──────────────────────────────
    scheduler_patience: int   = 5       # epochs without improvement before LR drop
    scheduler_factor:   float = 0.5     # LR = LR * factor on plateau
    min_lr:             float = 1e-6    # floor for LR reduction

    # ── Paths ──────────────────────────────────────────────────────────
    # Colab-shaped defaults -- __post_init__ below overrides these correctly
    # for Kaggle (or any platform) as long as IN_COLAB/IN_KAGGLE and
    # sequences_path are already defined in the session, which Phase 1's setup
    # cell guarantees.
    sequences_dir:   str = '/content/drive/My Drive/processed_data/sequences'
    checkpoint_dir:  str = '/content/drive/My Drive/processed_data/checkpoints'

    # ── Local staging (Task 7.1) ──────────────────────────────────────
    # X/y are shared across all 10 horizons since Phase 1's redesign, so staging
    # happens ONCE per train_all_horizons() call, not once per horizon. Drive FUSE
    # random reads are the likely dominant cost of training on Colab; copying the
    # (now much smaller, ~7GB total) sequence files to local disk once and reading
    # from there removes that bottleneck. Set use_local_staging=False to read
    # directly from Drive instead (e.g. if local disk space is tight). Defaults
    # to False automatically on Kaggle in __post_init__ below -- there's no
    # Drive mount there at all, so sequences are already local disk and staging
    # would just be a redundant local-to-local copy.
    use_local_staging: bool = True
    local_staging_dir: str  = '/content/_staged_sequences'

    # ── Horizons to train ─────────────────────────────────────────────
    horizons: List[int] = field(default_factory=lambda: list(range(1, 11)))

    def __post_init__(self):
        import builtins, gc as _gc

        # Platform-aware base paths take priority over the Colab-shaped class
        # defaults above. IN_COLAB/IN_KAGGLE are set once in Phase 1's platform
        # detection cell and don't change during a session.
        _in_kaggle = globals().get('IN_KAGGLE', False)
        if _in_kaggle:
            self.checkpoint_dir     = '/kaggle/working/processed_data/checkpoints'
            self.local_staging_dir  = '/kaggle/working/_staged_sequences'
            self.use_local_staging  = False

        # Bug fix: this lookup previously used __builtins__/builtins, which does
        # NOT reliably see notebook-level globals (confirmed by offline
        # execution test -- it silently failed to pick up sequences_path, and
        # was only ever masked by a later cell doing its own correct
        # globals().get(...) override right before training). globals() is the
        # correct way to check for a notebook-level variable from inside a
        # method defined at module/notebook scope.
        for _varname in ('sequences_path', 'sequences_dir'):
            _val = globals().get(_varname)
            if _val is not None:
                self.sequences_dir = str(_val)
                break

        # Auto-detect models_save_path / checkpoint_dir from session if available
        for _varname in ('models_save_path', 'checkpoint_dir', 'models_dir'):
            _val = globals().get(_varname)
            if _val is not None:
                self.checkpoint_dir = str(_val)
                break

        Path(self.checkpoint_dir).mkdir(parents=True, exist_ok=True)

    def display(self):
        print("=" * 50)
        print("TrainingConfig")
        print("=" * 50)
        print(f"  Model    : hidden={self.hidden_size}, layers={self.num_layers}, dropout={self.dropout}")
        print(f"  Training : batch={self.batch_size}, lr={self.learning_rate}, epochs={self.num_epochs}")
        print(f"  Stopping : patience={self.patience}, scheduler_patience={self.scheduler_patience}")
        print(f"  Horizons : {self.horizons}")
        print(f"  Seqs dir : {self.sequences_dir}")
        print(f"  Ckpt dir : {self.checkpoint_dir}")
        print(f"  Staging  : use_local_staging={self.use_local_staging}, dir={self.local_staging_dir}")
        print("=" * 50)

config = TrainingConfig()
config.display()

TrainingConfig
  Model    : hidden=128, layers=2, dropout=0.2
  Training : batch=64, lr=0.001, epochs=50
  Stopping : patience=10, scheduler_patience=5
  Horizons : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Seqs dir : /kaggle/working/processed_data/sequences
  Ckpt dir : /kaggle/working/processed_data/checkpoints
  Staging  : use_local_staging=False, dir=/kaggle/working/_staged_sequences


## Step 12: MmapSequenceDataset

The Phase 2 `SequenceDataset` loads the **entire array into RAM** in `__init__`.
For train X = 4.3 GB that causes an OOM crash in Colab.

`MmapSequenceDataset` uses `np.load(mmap_mode='r')`:
- The file is **memory-mapped** — no data is read at open time
- Each `__getitem__` reads only the rows it needs from disk
- The OS caches hot pages automatically
- Peak RAM = one batch (≈ 64 × 240 × 27 × 4 bytes = **1.7 MB**) not 4.3 GB

X is shared across all 10 horizons (Phase 1 writes it once per split, not once per
horizon). y is also written once per split as `(n, 10, 4)`; this class slices out the
column for the requested `horizon` at init time using basic indexing, which stays a
lazy memmap view rather than materializing all 10 horizons.

In [68]:
# Fix #1 hypothesis test (CPU delta target). OFF by default -- when False,
# __getitem__ below is byte-for-byte identical to prior behavior, so all
# 10 horizons / 4 targets are completely unaffected unless explicitly enabled.
CPU_DELTA_MODE = False
CPU_COL_IN_X   = None   # set to feature_cols.index('container_cpu_usage_seconds_total') before enabling
DELTA_MEAN     = None   # set to train-split mean of (future_cpu_real - last_cpu_real) before enabling
DELTA_STD      = None   # set to train-split std of the same, before enabling

class MmapSequenceDataset(Dataset):
    """
    Memory-mapped Dataset for large .npy sequence files.

    Uses np.load(mmap_mode='r') so the full array is NEVER loaded into RAM.
    Each __getitem__ call reads only the requested rows from disk.
    Safe for files larger than available RAM (e.g. 4 GB+ train X files).

    X is shared across all 10 horizons (Phase 1 writes it once per split). y is also
    written once per split with shape (n, 10, 4); this class slices out the column
    for `horizon` at init time using basic indexing, which stays a lazy memmap view
    (no data read) rather than materializing all 10 horizons.

    Parameters
    ----------
    X_path  : str   Path to sequences_X_{split}.npy
    y_path  : str   Path to sequences_y_{split}.npy (all horizons)
    horizon : int   Which horizon (1-10) to slice out of y
    """

    def __init__(self, X_path: str, y_path: str, horizon: int) -> None:
        if not Path(X_path).exists():
            raise FileNotFoundError(f"X file not found: {X_path}")
        if not Path(y_path).exists():
            raise FileNotFoundError(f"y file not found: {y_path}")

        # mmap_mode='r' -> file is mapped, not loaded
        self.X       = np.load(X_path, mmap_mode='r')          # (n, seq_len, n_feat) -- shared
        y_all        = np.load(y_path, mmap_mode='r')          # (n, 10, 4) -- shared
        # Basic indexing (single integer on axis 1) stays a memmap view -- no full read
        self.y       = y_all[:, horizon - 1, :]                 # (n, n_targets) -- this horizon only
        self.horizon = horizon
        self.X_path  = X_path
        self.y_path  = y_path

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        # np.array(...) copies the mmap slice into a regular array
        # This is required before converting to tensor
        x = torch.from_numpy(np.array(self.X[idx], dtype=np.float32))
        y = torch.from_numpy(np.array(self.y[idx], dtype=np.float32))
        if CPU_DELTA_MODE:
            # y[0] currently holds raw cpu_usage in z-score space. Convert to a
            # delta target, normalized by the delta's OWN (train-fit) mean/std
            # rather than reusing the raw value's stats -- deltas are much
            # smaller in magnitude than the raw counter, so reusing TARGET_STD[0]
            # would make this loss component's gradient negligible next to the
            # memory columns'.
            delta_real = (y[0] - x[-1, CPU_COL_IN_X]) * TARGET_STD[0]
            y[0] = (delta_real - DELTA_MEAN) / DELTA_STD
        return x, y

    def __repr__(self) -> str:
        return (f"MmapSequenceDataset("
                f"n={len(self.X)}, "
                f"seq_len={self.X.shape[1]}, "
                f"n_feat={self.X.shape[2]}, "
                f"horizon={self.horizon})")


def build_mmap_dataloaders(
    sequences_dir: str,
    horizon: int,
    batch_size: int = 64,
) -> dict:
    """
    Build train / val / test DataLoaders using memory-mapped arrays.
    Safe for files larger than available RAM.

    Parameters
    ----------
    sequences_dir : str   Path to .npy files.
    horizon       : int   Prediction horizon (1-10) -- sliced out of the shared y array.
    batch_size    : int   Samples per batch.

    Returns
    -------
    dict  {'train': DataLoader, 'val': DataLoader, 'test': DataLoader}
    """
    base       = Path(sequences_dir)
    split_cfg  = [('train', True), ('val', False), ('test', False)]
    loaders    = {}

    for split, shuffle in split_cfg:
        X_path = str(base / f"sequences_X_{split}.npy")
        y_path = str(base / f"sequences_y_{split}.npy")
        ds     = MmapSequenceDataset(X_path, y_path, horizon)
        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=0,           # 0 = safe for Colab + mmap
            pin_memory=torch.cuda.is_available(),
        )
        logger.info(f"[mmap] horizon={horizon} {split}: {len(ds):,} samples | {len(loaders[split])} batches")

    return loaders

def _stage_sequences_to_local(drive_dir: str, local_dir: str) -> str:
    """
    Copy the (shared, not per-horizon) sequence files from Drive to local Colab
    disk ONCE before training starts, so all 10 horizons read from local disk
    instead of paying Drive-FUSE random-read latency on every batch.

    Copies ONLY the exact filenames this pipeline's current sequence generator
    produces -- NOT a 'sequences_*.npy' wildcard. A wildcard glob here previously
    caused a real failure: if a Drive sequences/ folder still has leftover files
    from an OLDER run (the pre-fix generator wrote 'sequences_horizon_{h}_X_...'
    per horizon, ~69GB total), the wildcard would sweep those up too, alongside
    the new ~13GB shared files, and blow the local-disk space check for no reason
    ('need ~73GB, only 69GB free' even though the actual current data is ~13GB).
    Listing exact filenames makes this correct regardless of what old files are
    sitting in that Drive folder.

    Checks free local disk space first and raises rather than silently filling
    the disk -- Colab's local disk has previously filled up mid-run and produced
    truncated .npy files with a valid header but no body (EOFError, far from the
    real cause). Checking free space up front avoids repeating that failure mode.
    """
    import shutil as _shutil
    src = Path(drive_dir)
    dst = Path(local_dir)
    dst.mkdir(parents=True, exist_ok=True)

    expected_names = []
    for split in ('train', 'val', 'test'):
        expected_names += [
            f"sequences_X_{split}.npy",
            f"sequences_y_{split}.npy",
            f"sequences_containers_{split}.npy",
            f"sequences_metadata_{split}.json",
        ]

    files_to_copy = [src / name for name in expected_names if (src / name).exists()]
    missing = [name for name in expected_names if not (src / name).exists()]
    if missing:
        logger.warning(f"Expected sequence files missing under {src}: {missing} -- "
                        f"re-run Phase 1 if these are needed.")

    if not files_to_copy:
        logger.warning(f"No sequence files found under {src} to stage -- falling back to reading from Drive directly.")
        return str(src)

    total_bytes = sum(f.stat().st_size for f in files_to_copy)
    # Check free space on the ACTUAL target filesystem, not a hardcoded path --
    # '/content' doesn't exist on Kaggle, and a hardcoded path here would just
    # be wrong on any platform other than the one it was written for.
    free_bytes  = _shutil.disk_usage(str(dst)).free
    # Require 20% headroom beyond the copy itself -- never fill local disk to the brim.
    if total_bytes * 1.2 > free_bytes:
        raise RuntimeError(
            f"Not enough local disk space to stage sequences: need ~{total_bytes/1e9:.1f}GB "
            f"(+20% headroom), only {free_bytes/1e9:.1f}GB free on {dst}. "
            f"Set config.use_local_staging = False to read directly from Drive instead."
        )

    logger.info(f"Staging {len(files_to_copy)} files ({total_bytes/1e9:.2f}GB) "
                f"from Drive to local disk ({dst})...")
    for f in files_to_copy:
        _shutil.copy2(str(f), str(dst / f.name))
    logger.info(f"  ✓ Staging complete")
    return str(dst)


def _cleanup_local_staging(local_dir: str) -> None:
    """Remove the staged local copy after training -- never leave large files
    sitting on Colab's local disk across sessions (the same disk-full failure
    mode staging is meant to avoid, just deferred to the NEXT run instead)."""
    import shutil as _shutil
    p = Path(local_dir)
    if p.exists():
        _shutil.rmtree(str(p), ignore_errors=True)
        logger.info(f"  ✓ Cleaned up staged local sequences at {p}")


def _resave_checkpoint_with_results(ckpt_path, seed, run_id, val_metrics, test_metrics) -> None:
    """save_checkpoint() runs mid-training, before test-set results exist -- this
    reloads the just-saved best checkpoint and attaches the final val/test metrics,
    seed, and run_id onto it without needing a second full save (which would require
    re-passing optimizer state that's no longer needed once training is done)."""
    ckpt = torch.load(str(ckpt_path), map_location=device)
    ckpt['val_metrics']  = val_metrics
    ckpt['test_metrics'] = test_metrics
    ckpt['seed']         = seed
    ckpt['run_id']       = run_id
    torch.save(ckpt, str(ckpt_path))


print("_stage_sequences_to_local() defined.")
print("_cleanup_local_staging() defined.")
print("_resave_checkpoint_with_results() defined.")

_stage_sequences_to_local() defined.
_cleanup_local_staging() defined.
_resave_checkpoint_with_results() defined.


## Step 13: EarlyStopping

Monitors validation loss each epoch.
If it does not improve for `patience` consecutive epochs, sets `early_stop = True`.
The training loop checks this flag and breaks.

In [69]:
class EarlyStopping:
    """
    Stops training when validation loss stops improving.

    Parameters
    ----------
    patience  : int    Epochs to wait before stopping. Default 10.
    min_delta : float  Minimum improvement to count as improvement. Default 1e-6.

    Usage
    -----
    es = EarlyStopping(patience=10)
    for epoch in ...:
        val_loss = validate(...)
        es(val_loss)
        if es.early_stop:
            break
    """

    def __init__(self, patience: int = 10, min_delta: float = 1e-6) -> None:
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float('inf')
        self.counter    = 0
        self.early_stop = False

    def __call__(self, val_loss: float) -> bool:
        """
        Call after each validation step.

        Returns True if this epoch has a new best loss.
        """
        if val_loss < self.best_loss - self.min_delta:
            # Improvement — reset counter
            self.best_loss = val_loss
            self.counter   = 0
            return True     # new best
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False    # no improvement

    def __repr__(self) -> str:
        return (f"EarlyStopping(patience={self.patience}, "
                f"counter={self.counter}, best={self.best_loss:.6f})")

print("EarlyStopping defined.")

EarlyStopping defined.


## Step 14: train_one_epoch() and validate()

`train_one_epoch()`:
- Sets model to **train mode** (dropout active)
- Iterates all batches, computes MSELoss, backpropagates
- Clips gradients to prevent exploding gradients
- Returns mean loss over the full epoch

`validate()`:
- Sets model to **eval mode** (dropout disabled)
- Runs with `torch.no_grad()` — no gradient computation, faster and less memory
- Returns mean loss over the validation set

In [70]:
def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    config:    TrainingConfig,
) -> float:
    """
    One complete pass through the training DataLoader.

    Steps per batch
    ---------------
    1. Move X, y to device (GPU if available)
    2. Zero gradients
    3. Forward pass -> predictions
    4. Compute MSELoss
    5. Backward pass -> gradients
    6. Clip gradients (prevents exploding gradients in RNNs)
    7. Optimizer step -> update weights

    Returns
    -------
    float : mean training loss over the epoch (normalized/z-score space -- this
            is the space the model is actually optimized in)
    """
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        predictions = model(X_batch)               # (batch, 4)
        loss        = criterion(predictions, y_batch)

        loss.backward()

        # Gradient clipping — essential for RNNs to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.grad_clip)

        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    return total_loss / len(loader.dataset)


def validate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
) -> dict:
    """
    Evaluation pass — no weight updates, no gradient computation.

    Returns a dict:
      'loss'            : float -- mean MSE in NORMALIZED (z-score) space. This is
                           what the LR scheduler and early stopping watch, since
                           that's the space the model is actually trained in.
      'mae_per_metric'  : {target_name: MAE in REAL units (CPU-seconds / bytes)}
      'rmse_per_metric' : {target_name: RMSE in REAL units}
      'mape_per_metric' : {target_name: {'mape': float %%, 'n_excluded_near_zero': int}}
      'mae_mean' / 'rmse_mean' / 'mape_mean' : averaged across the 4 targets

    Predictions/targets are inverse z-score normalized (TARGET_MEAN/TARGET_STD,
    loaded from Phase 1's normalization_stats_train.json) before computing
    MAE/RMSE/MAPE -- reporting MAE/RMSE directly in normalized units is not
    interpretable on its own (a "MAE of 0.05" means nothing without knowing it's
    0.05 standard deviations, not 0.05 bytes).

    MAPE uses a floored denominator (epsilon=1e-6): a near-zero actual reading
    (e.g. an idle container) would otherwise blow MAPE up towards infinity. Points
    below the floor are excluded from the mean and counted in n_excluded_near_zero
    rather than silently distorting the average.
    """
    model.eval()
    total_loss  = 0.0
    all_preds   = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            predictions = model(X_batch)
            loss        = criterion(predictions, y_batch)

            total_loss += loss.item() * len(X_batch)
            all_preds.append(predictions.cpu())
            all_targets.append(y_batch.cpu())

    n_samples = len(loader.dataset)
    mean_loss = total_loss / n_samples

    preds_norm   = torch.cat(all_preds,   dim=0).numpy()   # (n, 4) -- normalized space
    targets_norm = torch.cat(all_targets, dim=0).numpy()   # (n, 4) -- normalized space

    # Inverse z-score: real = normalized * std + mean
    preds_real   = preds_norm   * TARGET_STD + TARGET_MEAN
    targets_real = targets_norm * TARGET_STD + TARGET_MEAN

    abs_err = np.abs(preds_real - targets_real)   # (n, 4) real units
    sq_err  = (preds_real - targets_real) ** 2    # (n, 4) real units

    epsilon    = 1e-6
    near_zero  = np.abs(targets_real) < epsilon
    safe_denom = np.where(near_zero, epsilon, np.abs(targets_real))
    ape        = abs_err / safe_denom * 100.0
    ape_masked = np.where(near_zero, np.nan, ape)

    mae_per_metric  = {}
    rmse_per_metric = {}
    mape_per_metric = {}
    for i, name in enumerate(TARGET_NAMES):
        mae_per_metric[name]  = float(abs_err[:, i].mean())
        rmse_per_metric[name] = float(np.sqrt(sq_err[:, i].mean()))
        col_ape    = ape_masked[:, i]
        n_excluded = int(np.isnan(col_ape).sum())
        mape_per_metric[name] = {
            'mape': float(np.nanmean(col_ape)) if n_excluded < len(col_ape) else float('nan'),
            'n_excluded_near_zero': n_excluded,
        }

    return {
        'loss':            mean_loss,
        'mae_per_metric':  mae_per_metric,
        'rmse_per_metric': rmse_per_metric,
        'mape_per_metric': mape_per_metric,
        'mae_mean':  float(np.mean(list(mae_per_metric.values()))),
        'rmse_mean': float(np.mean(list(rmse_per_metric.values()))),
        'mape_mean': float(np.nanmean([m['mape'] for m in mape_per_metric.values()])),
    }

print("train_one_epoch() defined.")
print("validate() defined.")

train_one_epoch() defined.
validate() defined.


## Step 15: save_checkpoint()

Saves the best model state to Google Drive.

Checkpoint contains:
- `model_state_dict` — all learned weights
- `optimizer_state_dict` — optimizer momentum/state (for resuming)
- `epoch` — which epoch this was saved at
- `val_loss` — best validation loss achieved
- `config` — full hyperparameter snapshot

In [71]:
def save_checkpoint(
    model:       nn.Module,
    optimizer:   torch.optim.Optimizer,
    epoch:       int,
    val_loss:    float,
    horizon:     int,
    config:      TrainingConfig,
    val_metrics: dict = None,
) -> str:
    """
    Save the best model checkpoint to Google Drive.

    File name: gru_horizon_{horizon}_best.pt
    Overwrites the previous best for this horizon.

    Checkpoint schema (Task 5.4 / 6.3 additions over the original version):
      - val_metrics : the full real-unit MAE/RMSE/MAPE dict from validate(), not
                       just the normalized-space scalar loss. May be None here --
                       train_all_horizons() attaches the FINAL val/test metrics
                       (evaluated from this exact checkpoint) after training ends,
                       via _resave_checkpoint_with_results().
      - seed / run_id / test_metrics : added later by
                       _resave_checkpoint_with_results(), not at save time here,
                       since test-set results don't exist until training is done.

    Returns
    -------
    str : full path to saved checkpoint file
    """
    ckpt_dir = Path(config.checkpoint_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    filepath = ckpt_dir / f"gru_horizon_{horizon}_best.pt"

    checkpoint = {
        'epoch':                epoch,
        'horizon':               horizon,
        'val_loss':              val_loss,
        'val_metrics':           val_metrics,
        'model_state_dict':      model.state_dict(),
        'optimizer_state_dict':  optimizer.state_dict(),
        'config': {
            'input_size':  config.input_size,
            'hidden_size': config.hidden_size,
            'num_layers':  config.num_layers,
            'output_size': config.output_size,
            'dropout':     config.dropout,
        },
    }

    torch.save(checkpoint, str(filepath))
    return str(filepath)


def load_checkpoint(filepath: str, config: TrainingConfig) -> nn.Module:
    """
    Load a saved checkpoint and return the model with restored weights.

    Parameters
    ----------
    filepath : str           Path to .pt checkpoint file.
    config   : TrainingConfig  Used to rebuild the model architecture.

    Returns
    -------
    nn.Module : model with loaded weights, moved to device, in eval mode.
    """
    ckpt  = torch.load(filepath, map_location=device)
    model = GRUModel(
        input_size=config.input_size,
        hidden_size=config.hidden_size,
        num_layers=config.num_layers,
        output_size=config.output_size,
        dropout=config.dropout,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    logger.info(f"Loaded checkpoint: epoch={ckpt['epoch']} val_loss={ckpt['val_loss']:.6f}")
    return model

print("save_checkpoint() defined.")
print("load_checkpoint() defined.")

save_checkpoint() defined.
load_checkpoint() defined.


## Step 16: train_model()

Full training loop for **one horizon**:

```
for each epoch:
    1. train_one_epoch()  -> train_loss
    2. validate()         -> val_loss
    3. scheduler.step()   -> reduce LR if val_loss plateaued
    4. EarlyStopping()    -> if new best: save checkpoint
                          -> if no improvement x patience: stop
```

Returns training history (loss curves) and best val loss.

In [72]:
def train_model(
    model:        nn.Module,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    config:       TrainingConfig,
    horizon:      int,
) -> tuple:
    """
    Full training loop for one GRU model (one horizon).

    Parameters
    ----------
    model        : GRUModel instance (already moved to device)
    train_loader : DataLoader for training split
    val_loader   : DataLoader for validation split
    config       : TrainingConfig with all hyperparameters
    horizon      : int, used for checkpoint file naming

    Returns
    -------
    history       : dict with:
                     'train_loss' : list[float], normalized-space MSE per epoch
                     'val_loss'   : list[float], normalized-space MSE per epoch
                                    (this is what scheduler/early-stopping watch)
                     'val_metrics': list[dict], the full validate() return value
                                    per epoch (real-unit MAE/RMSE/MAPE)
    best_val_loss : float, lowest NORMALIZED validation loss achieved
    """
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

    # ReduceLROnPlateau: reduce LR by factor when val_loss stops improving
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=config.scheduler_factor,
        patience=config.scheduler_patience,
        min_lr=config.min_lr,
    )

    early_stopping = EarlyStopping(patience=config.patience)

    history = {'train_loss': [], 'val_loss': [], 'val_metrics': []}
    best_val_loss = float('inf')
    start_time    = time.time()

    for epoch in range(1, config.num_epochs + 1):

        # ── Train ────────────────────────────────────────────────────
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, config)

        # ── Validate ─────────────────────────────────────────────────
        val_metrics = validate(model, val_loader, criterion)
        val_loss    = val_metrics['loss']   # normalized-space MSE -- drives scheduler/ES

        # ── LR scheduler ─────────────────────────────────────────────
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        # ── Record ───────────────────────────────────────────────────
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_metrics'].append(val_metrics)

        # ── Early stopping + checkpoint ───────────────────────────────
        is_best = early_stopping(val_loss)
        if is_best:
            best_val_loss = val_loss
            ckpt_path = save_checkpoint(model, optimizer, epoch, val_loss, horizon, config, val_metrics)

        # ── Logging (every 5 epochs and first epoch) ──────────────────
        if epoch == 1 or epoch % 5 == 0:
            elapsed = time.time() - start_time
            print(
                f"  Epoch {epoch:3d}/{config.num_epochs}"
                f" | train={train_loss:.6f}"
                f" | val={val_loss:.6f}"
                f" | val_MAPE={val_metrics['mape_mean']:.2f}%"
                f" | lr={current_lr:.2e}"
                f" | patience={early_stopping.counter}/{config.patience}"
                f" | {elapsed:.0f}s"
            )

        # ── Stop if no improvement ────────────────────────────────────
        if early_stopping.early_stop:
            print(f"  Early stopping triggered at epoch {epoch}")
            break

    total_time = time.time() - start_time
    print(f"  Done: {len(history['train_loss'])} epochs | best_val_loss={best_val_loss:.6f} | {total_time:.0f}s")

    return history, best_val_loss

print("train_model() defined.")

train_model() defined.


## Step 17: train_all_horizons()

Loops over all configured horizons (1–10).
For each horizon:
1. Loads train + val DataLoaders (memory-mapped)
2. Creates a fresh `GRUModel`
3. Calls `train_model()`
4. Records results
5. Frees model from memory before next horizon

In [73]:
def train_all_horizons(config: TrainingConfig) -> dict:
    """
    Train one independent GRU model per prediction horizon.

    For each horizon:
      - Re-seeds (42 + horizon) BEFORE building the model -- each horizon's init
        and shuffle sequence is independently reproducible, not dependent on the
        RNG state left behind by every horizon that ran before it.
      - Loads sequences from a LOCAL staged copy of Phase 1's output (staged once
        for the whole run, not per horizon, since X/y are now shared across all
        10 horizons) instead of reading directly from Drive on every batch.
      - Creates a fresh GRUModel with Xavier + Orthogonal weight init.
      - Trains with early stopping, LR scheduling, and mid-training checkpointing.
      - Reloads the BEST checkpoint (not whatever the in-memory model happens to be
        -- early stopping's patience window keeps training several epochs past the
        best one, so the in-memory model is usually NOT the best model by the time
        the loop ends) and evaluates it on val AND the held-out test split.
      - Attaches seed/run_id/final val+test metrics onto the checkpoint file.
      - Frees memory before moving to the next horizon.

    Parameters
    ----------
    config : TrainingConfig

    Returns
    -------
    dict  {horizon: {best_val_loss, val_mae_mean, val_rmse_mean, val_mape_mean,
                      test_mae_mean, test_rmse_mean, test_mape_mean,
                      mape_target_pass, epochs_trained, seed}}
    """
    run_id = time.strftime('%Y%m%d_%H%M%S')
    results = {}
    overall_start = time.time()
    MAPE_TARGET = 10.0  # proposal's stated success criterion: +/-10% MAPE

    # ── Stage shared sequence files to local disk ONCE (Task 7.1) ──────────────
    # X and y are shared across all 10 horizons since Phase 1's redesign, so this
    # only needs to happen once per run, not once per horizon.
    staged_dir = config.sequences_dir
    if config.use_local_staging:
        staged_dir = _stage_sequences_to_local(config.sequences_dir, config.local_staging_dir)

    try:
        for horizon in config.horizons:
            sep = "=" * 58
            print(f"\n{sep}")
            print(f"HORIZON {horizon}/10  ({horizon * 15} seconds ahead)  [run_id={run_id}]")
            print(sep)

            # ── Re-seed per horizon (Task 6.1) ──────────────────────────────
            horizon_seed = 42 + horizon
            set_seed(horizon_seed)

            # ── Load data (from local staged copy when staging is enabled) ──
            try:
                loaders = build_mmap_dataloaders(
                    staged_dir,
                    horizon=horizon,
                    batch_size=config.batch_size,
                )
            except FileNotFoundError as e:
                print(f"  SKIPPED — file not found: {e}")
                continue

            print(f"  Train batches : {len(loaders['train'])}")
            print(f"  Val batches   : {len(loaders['val'])}")
            print(f"  Test batches  : {len(loaders['test'])}")

            # ── Build fresh model ─────────────────────────────────────────
            model = GRUModel(
                input_size=config.input_size,
                hidden_size=config.hidden_size,
                num_layers=config.num_layers,
                output_size=config.output_size,
                dropout=config.dropout,
            ).to(device)

            # ── Train ────────────────────────────────────────────────────
            history, best_val_loss = train_model(
                model, loaders['train'], loaders['val'], config, horizon
            )

            # ── Evaluate the BEST checkpoint, not the in-memory model (Task 5.3) ──
            ckpt_path  = Path(config.checkpoint_dir) / f"gru_horizon_{horizon}_best.pt"
            best_model = load_checkpoint(str(ckpt_path), config)
            criterion  = nn.MSELoss()
            val_metrics_best  = validate(best_model, loaders['val'],  criterion)
            test_metrics_best = validate(best_model, loaders['test'], criterion)

            # Attach seed/run_id/final metrics onto the checkpoint (Task 5.4 / 6.3)
            _resave_checkpoint_with_results(
                ckpt_path, seed=horizon_seed, run_id=run_id,
                val_metrics=val_metrics_best, test_metrics=test_metrics_best,
            )

            mape_pass = val_metrics_best['mape_mean'] <= MAPE_TARGET

            # ── Record results ────────────────────────────────────────────
            results[horizon] = {
                'best_val_loss':    best_val_loss,
                'val_mae_mean':     val_metrics_best['mae_mean'],
                'val_rmse_mean':    val_metrics_best['rmse_mean'],
                'val_mape_mean':    val_metrics_best['mape_mean'],
                'test_mae_mean':    test_metrics_best['mae_mean'],
                'test_rmse_mean':   test_metrics_best['rmse_mean'],
                'test_mape_mean':   test_metrics_best['mape_mean'],
                'mape_target_pass': mape_pass,
                'epochs_trained':   len(history['train_loss']),
                'seed':             horizon_seed,
            }

            print(f"  Horizon {horizon} | val_MAPE={val_metrics_best['mape_mean']:.2f}%  "
                  f"test_MAPE={test_metrics_best['mape_mean']:.2f}%  "
                  f"[{'PASS' if mape_pass else 'FAIL'} vs {MAPE_TARGET:.0f}% target]")

            # ── Free memory before next horizon ───────────────────────────
            del model, best_model, loaders, history
            torch.cuda.empty_cache()
            gc.collect()
    finally:
        # Always clean up staged local files, even if a horizon raised (Task 7.2) --
        # never leave a large local copy sitting around for the next Colab session.
        if config.use_local_staging:
            _cleanup_local_staging(config.local_staging_dir)

    total_time = time.time() - overall_start

    # ── Summary (Task 8.2) ───────────────────────────────────────────────────
    print(f"\n{'='*58}")
    print("TRAINING COMPLETE — Summary")
    print(f"{'='*58}")
    print(f"  {'Horizon':>8}  {'ValMAPE%':>9}  {'TestMAPE%':>10}  {'Epochs':>7}  {'Target':>7}")
    print(f"  {'-'*8}  {'-'*9}  {'-'*10}  {'-'*7}  {'-'*7}")
    for h, r in sorted(results.items()):
        print(f"  {h:>8}  {r['val_mape_mean']:>9.2f}  {r['test_mape_mean']:>10.2f}  "
              f"{r['epochs_trained']:>7}  {'PASS' if r['mape_target_pass'] else 'FAIL':>7}")
    print(f"{'='*58}")
    print(f"  Total wall-clock time: {total_time/60:.1f} min ({total_time:.0f}s)  -- "
          f"replaces any earlier unverified runtime estimate with a real measurement.")

    # Save summary to Drive (Task 5.4)
    summary_path = Path(config.checkpoint_dir) / 'training_summary.json'
    with open(str(summary_path), 'w') as f:
        json_lib.dump({'run_id': run_id, 'total_time_sec': total_time, 'mape_target': MAPE_TARGET,
                       'results': results}, f, indent=2)
    print(f"\nSummary saved: {summary_path}")

    return results

print("train_all_horizons() defined.")

train_all_horizons() defined.


## Step 17b: Cross-Phase Integration Check

Pre-flight check run right before the expensive 10-horizon training loop starts.
Confirms the constants Phase 1 (data), Phase 2 (`GRUModel`), and Phase 3
(`TrainingConfig`, `TARGET_NAMES`/`TARGET_MEAN`/`TARGET_STD`) each assume about
shapes actually agree with each other -- and, where Phase 1 metadata is available
in this session, agree with what Phase 1 actually produced. A silent mismatch here
(e.g. Phase 1 producing 26 features instead of 27 after an edit) would otherwise
only surface as a confusing shape-mismatch crash partway through hour 1 of training.

In [74]:
print("="*70)
print("CROSS-PHASE INTEGRATION CONSISTENCY CHECK")
print("="*70)

_checks = []

# Compare against REAL Phase 1 output metadata when it's available in this session,
# not just against hardcoded constants -- catches a Phase 1 edit that silently
# changed the feature count without also updating Phase 2/3's assumptions.
_meta_files = list(Path(config.sequences_dir).glob('sequences_metadata_*.json'))
if _meta_files:
    with open(_meta_files[0]) as _f:
        _meta = json.load(_f)
    _actual_n_features = _meta['n_features']
    _actual_lookback   = _meta['lookback_window']
    _actual_horizons   = _meta['horizons']
    print(f"  Using real Phase 1 metadata from: {_meta_files[0].name}")
else:
    _actual_n_features = _actual_lookback = _actual_horizons = None
    print("  (no Phase 1 metadata found yet at config.sequences_dir -- "
          "checking hardcoded constants against each other only, not against real output)")

_probe_model = GRUModel()  # defaults only, just to inspect the wired-up shapes

_checks.append(("GRUModel.input_size default == 27", _probe_model.input_size == 27,
                 f"got {_probe_model.input_size}"))
if _actual_n_features is not None:
    _checks.append(("GRUModel.input_size matches Phase 1's actual n_features",
                     _probe_model.input_size == _actual_n_features,
                     f"model={_probe_model.input_size}  phase1={_actual_n_features}"))
    _checks.append(("TrainingConfig.input_size matches Phase 1's actual n_features",
                     config.input_size == _actual_n_features,
                     f"config={config.input_size}  phase1={_actual_n_features}"))

_checks.append(("GRUModel.output_size == len(TARGET_NAMES)",
                 _probe_model.output_size == len(TARGET_NAMES),
                 f"model={_probe_model.output_size}  TARGET_NAMES={len(TARGET_NAMES)}"))
_checks.append(("TARGET_NAMES and TARGET_COLUMNS same length",
                 len(TARGET_NAMES) == len(TARGET_COLUMNS),
                 f"{len(TARGET_NAMES)} vs {len(TARGET_COLUMNS)}"))
_checks.append(("TrainingConfig.output_size == len(TARGET_NAMES)",
                 config.output_size == len(TARGET_NAMES),
                 f"config={config.output_size}  TARGET_NAMES={len(TARGET_NAMES)}"))
_checks.append(("TARGET_MEAN/TARGET_STD length == output_size",
                 len(TARGET_MEAN) == config.output_size == len(TARGET_STD),
                 f"MEAN={len(TARGET_MEAN)}  STD={len(TARGET_STD)}  output_size={config.output_size}"))

if _actual_horizons is not None:
    _checks.append(("config.horizons is a subset of what Phase 1 actually generated",
                     set(config.horizons).issubset(set(_actual_horizons)),
                     f"config.horizons={config.horizons}  phase1_horizons={_actual_horizons}"))
    _checks.append(("lookback_window Phase 1 used == 240 (GRUModel/roadmap assumption)",
                     _actual_lookback == 240,
                     f"phase1_lookback={_actual_lookback}"))

del _probe_model

print(f"\n  {'CHECK':<52} {'RESULT':<6}  EVIDENCE")
print(f"  {'-'*52} {'-'*6}  {'-'*36}")
_n_pass = 0
for name, passed, evidence in _checks:
    print(f"  {name:<52} {'PASS' if passed else 'FAIL':<6}  {evidence}")
    _n_pass += int(passed)

print(f"\n  {_n_pass}/{len(_checks)} checks passed")
if _n_pass == len(_checks):
    print("  ✅ Phase 1 -> Phase 2 -> Phase 3 constants are consistent. Safe to start training.")
else:
    print("  ⚠ Mismatch found — fix before calling train_all_horizons(); a mismatch here "
          "would otherwise only surface as a shape-mismatch crash partway through training.")
    raise AssertionError(f"{len(_checks) - _n_pass} cross-phase consistency check(s) failed -- see table above")

2026-07-24 11:45:23,668 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876


CROSS-PHASE INTEGRATION CONSISTENCY CHECK
  Using real Phase 1 metadata from: sequences_metadata_train.json

  CHECK                                                RESULT  EVIDENCE
  ---------------------------------------------------- ------  ------------------------------------
  GRUModel.input_size default == 27                    PASS    got 27
  GRUModel.input_size matches Phase 1's actual n_features PASS    model=27  phase1=27
  TrainingConfig.input_size matches Phase 1's actual n_features PASS    config=27  phase1=27
  GRUModel.output_size == len(TARGET_NAMES)            PASS    model=4  TARGET_NAMES=4
  TARGET_NAMES and TARGET_COLUMNS same length          PASS    4 vs 4
  TrainingConfig.output_size == len(TARGET_NAMES)      PASS    config=4  TARGET_NAMES=4
  TARGET_MEAN/TARGET_STD length == output_size         PASS    MEAN=4  STD=4  output_size=4
  config.horizons is a subset of what Phase 1 actually generated PASS    config.horizons=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  phase1_hori

## Step 18: Run Training

Starts training all 10 horizon models.
Checkpoints saved to:
```
/content/drive/My Drive/processed_data/checkpoints/gru_horizon_N_best.pt
```
Training summary saved to:
```
/content/drive/My Drive/processed_data/checkpoints/training_summary.json
```

> **Tip:** You can train a subset first to test:
> `config.horizons = [1]`  — trains only horizon 1
> `config.horizons = [1, 5, 10]`  — trains 3 horizons

In [75]:
# ── Override config paths from session variables if they exist ──────────────
import builtins as _b

_seq = (globals().get('sequences_path')
        or globals().get('sequences_dir')
        or getattr(_b, 'sequences_path', None))

_ckpt = (globals().get('models_save_path')
         or globals().get('checkpoint_dir')
         or globals().get('models_dir')
         or getattr(_b, 'models_save_path', None))

if _seq is not None:
    config.sequences_dir = str(_seq)
if _ckpt is not None:
    config.checkpoint_dir = str(_ckpt)
    Path(config.checkpoint_dir).mkdir(parents=True, exist_ok=True)

# ── Optional: reduce horizons for a quick test ──────────────────────────────
# config.horizons = [1]         # single horizon test
# config.horizons = [1, 5, 10]  # 3 horizons
# config.horizons = list(range(1, 11))  # all 10 (default)

config.display()

print("\nStarting training...\n")
results = train_all_horizons(config)


2026-07-24 11:45:23,679 - INFO - [mmap] horizon=1 train: 149,916 samples | 2343 batches
2026-07-24 11:45:23,681 - INFO - [mmap] horizon=1 val: 20,348 samples | 318 batches
2026-07-24 11:45:23,683 - INFO - [mmap] horizon=1 test: 20,356 samples | 319 batches
2026-07-24 11:45:23,691 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876


TrainingConfig
  Model    : hidden=128, layers=2, dropout=0.2
  Training : batch=64, lr=0.001, epochs=50
  Stopping : patience=10, scheduler_patience=5
  Horizons : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Seqs dir : /kaggle/working/processed_data/sequences
  Ckpt dir : /kaggle/working/processed_data/checkpoints
  Staging  : use_local_staging=False, dir=/kaggle/working/_staged_sequences

Starting training...


HORIZON 1/10  (15 seconds ahead)  [run_id=20260724_114523]
  Train batches : 2343
  Val batches   : 318
  Test batches  : 319
  Epoch   1/50 | train=0.020514 | val=0.010332 | val_MAPE=2.68% | lr=1.00e-03 | patience=0/10 | 23s
  Epoch   5/50 | train=0.012240 | val=0.006781 | val_MAPE=1.13% | lr=1.00e-03 | patience=2/10 | 113s
  Epoch  10/50 | train=0.010028 | val=0.004463 | val_MAPE=1.05% | lr=1.00e-03 | patience=3/10 | 225s
  Epoch  15/50 | train=0.007100 | val=0.006550 | val_MAPE=1.28% | lr=5.00e-04 | patience=8/10 | 336s


2026-07-24 11:51:45,016 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876
2026-07-24 11:51:45,019 - INFO - Loaded checkpoint: epoch=7 val_loss=0.001992


  Early stopping triggered at epoch 17
  Done: 17 epochs | best_val_loss=0.001992 | 381s
  Horizon 1 | val_MAPE=0.92%  test_MAPE=1.19%  [PASS vs 10% target]


2026-07-24 11:51:48,135 - INFO - [mmap] horizon=2 train: 149,916 samples | 2343 batches
2026-07-24 11:51:48,137 - INFO - [mmap] horizon=2 val: 20,348 samples | 318 batches
2026-07-24 11:51:48,138 - INFO - [mmap] horizon=2 test: 20,356 samples | 319 batches
2026-07-24 11:51:48,145 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876


...

  Early stopping triggered at epoch 16
  Done: 16 epochs | best_val_loss=0.003118 | 358s
  Horizon 9 | val_MAPE=1.00%  test_MAPE=1.32%  [PASS vs 10% target]


2026-07-24 12:49:59,464 - INFO - [mmap] horizon=10 train: 149,916 samples | 2343 batches
2026-07-24 12:49:59,465 - INFO - [mmap] horizon=10 val: 20,348 samples | 318 batches
2026-07-24 12:49:59,467 - INFO - [mmap] horizon=10 test: 20,356 samples | 319 batches
2026-07-24 12:49:59,474 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876



HORIZON 10/10  (150 seconds ahead)  [run_id=20260724_114523]
  Train batches : 2343
  Val batches   : 318
  Test batches  : 319
  Epoch   1/50 | train=0.047712 | val=0.004707 | val_MAPE=1.67% | lr=1.00e-03 | patience=0/10 | 23s
  Epoch   5/50 | train=0.032664 | val=0.005305 | val_MAPE=1.18% | lr=1.00e-03 | patience=1/10 | 112s
  Epoch  10/50 | train=0.027833 | val=0.016724 | val_MAPE=1.72% | lr=5.00e-04 | patience=6/10 | 223s
  Epoch  15/50 | train=0.022645 | val=0.004751 | val_MAPE=1.30% | lr=5.00e-04 | patience=2/10 | 335s
  Epoch  20/50 | train=0.020388 | val=0.007537 | val_MAPE=1.28% | lr=2.50e-04 | patience=7/10 | 447s


2026-07-24 12:58:33,235 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876
2026-07-24 12:58:33,237 - INFO - Loaded checkpoint: epoch=13 val_loss=0.002666


  Early stopping triggered at epoch 23
  Done: 23 epochs | best_val_loss=0.002666 | 514s
  Horizon 10 | val_MAPE=1.05%  test_MAPE=1.49%  [PASS vs 10% target]

TRAINING COMPLETE — Summary
   Horizon   ValMAPE%   TestMAPE%   Epochs   Target
  --------  ---------  ----------  -------  -------
         1       0.92        1.19       17     PASS
         2       0.68        1.17       23     PASS
         3       0.99        1.31       17     PASS
         4       1.05        1.26       13     PASS
         5       0.97        1.67       23     PASS
         6       1.05        1.36       16     PASS
         7       1.47        1.62       14     PASS
         8       1.00        1.55       33     PASS
         9       1.00        1.32       16     PASS
        10       1.05        1.49       23     PASS
  Total wall-clock time: 73.2 min (4392s)  -- replaces any earlier unverified runtime estimate with a real measurement.

Summary saved: /kaggle/working/processed_data/checkpoints/training_s

## Step 19: Phase 3 Verification (Validation Matrix)

Same PASS/FAIL evidence-based approach as Phase 1's "Verify Sequences" and Phase 2's
"Smoke Test" -- checks real files/values produced by the run above, not just that the
code executed without raising. Run this after `train_all_horizons(config)` completes.

Checks:
1. Every configured horizon has a checkpoint file, and it's loadable with the expected keys
2. Each checkpoint's recorded `val_loss` matches the value in `results` (Task 8.4)
3. Each horizon's val MAPE against the proposal's +/-10% success target (Task 5.5)
4. `training_summary.json` was saved to Drive

In [76]:
print("="*70)
print("PHASE 3 VALIDATION MATRIX")
print("="*70)

_checks = []  # (name, passed: bool, evidence: str)
_ckpt_dir = Path(config.checkpoint_dir)

for h in config.horizons:
    _p = _ckpt_dir / f"gru_horizon_{h}_best.pt"
    if not _p.exists():
        _checks.append((f"horizon {h}: checkpoint exists", False, f"missing: {_p}"))
        continue

    try:
        _ckpt = torch.load(str(_p), map_location='cpu')
        _required_keys = {'epoch', 'horizon', 'val_loss', 'model_state_dict', 'config'}
        _missing = _required_keys - set(_ckpt.keys())
        if _missing:
            _checks.append((f"horizon {h}: checkpoint has required keys", False, f"missing keys: {_missing}"))
        else:
            _checks.append((f"horizon {h}: checkpoint loadable", True,
                             f"epoch={_ckpt['epoch']} val_loss={_ckpt['val_loss']:.6f}"))

        # Cross-check recorded val_loss against the in-memory training results (Task 8.4)
        if 'results' in dir() and h in results and 'best_val_loss' in results[h]:
            _match = abs(_ckpt['val_loss'] - results[h]['best_val_loss']) < 1e-6
            _checks.append((f"horizon {h}: checkpoint val_loss matches history", _match,
                             f"ckpt={_ckpt['val_loss']:.6f}  history={results[h]['best_val_loss']:.6f}"))
    except Exception as e:
        _checks.append((f"horizon {h}: checkpoint loadable", False, f"torch.load failed: {e}"))

# MAPE vs the proposal's +/-10% target, per horizon (Task 5.5)
if 'results' in dir():
    for h, r in sorted(results.items()):
        _checks.append((f"horizon {h}: val MAPE <= 10% target", r.get('mape_target_pass', False),
                         f"val_MAPE={r.get('val_mape_mean', float('nan')):.2f}%"))

# WBS Task 6.2: test-set MAPE was previously never surfaced in this matrix --
# V4/V5 showed test MAPE running ~10x worse than val MAPE on every horizon and
# it went unnoticed until a dedicated investigation. Report it explicitly from
# now on, plus an explicit val/test gap flag, so this can never again go
# unmeasured for multiple project versions.
if 'results' in dir():
    for h, r in sorted(results.items()):
        _test_mape = r.get('test_mape_mean', float('nan'))
        _checks.append((f"horizon {h}: test MAPE <= 10% target", _test_mape <= 10.0,
                         f"test_MAPE={_test_mape:.2f}%"))
    for h, r in sorted(results.items()):
        _val_mape = r.get('val_mape_mean', float('nan'))
        _test_mape = r.get('test_mape_mean', float('nan'))
        _ratio = (_test_mape / _val_mape) if _val_mape else float('nan')
        _gap_ok = _ratio <= 2.0  # flag when test is more than 2x worse than val
        _checks.append((f"horizon {h}: test/val MAPE ratio <= 2x", _gap_ok,
                         f"val={_val_mape:.2f}%  test={_test_mape:.2f}%  ratio={_ratio:.1f}x"))

# training_summary.json persisted
_summary_path = _ckpt_dir / 'training_summary.json'
_checks.append(("training_summary.json saved", _summary_path.exists(), str(_summary_path)))

print(f"\n  {'CHECK':<48} {'RESULT':<6}  EVIDENCE")
print(f"  {'-'*48} {'-'*6}  {'-'*32}")
_n_pass = 0
for name, passed, evidence in _checks:
    print(f"  {name:<48} {'PASS' if passed else 'FAIL':<6}  {evidence}")
    _n_pass += int(passed)

print(f"\n  {_n_pass}/{len(_checks)} checks passed")
if _n_pass == len(_checks):
    print("  ✅ Phase 3 fully verified.")
else:
    print("  ⚠ Some checks failed — see FAIL rows above before proceeding.")

PHASE 3 VALIDATION MATRIX

  CHECK                                            RESULT  EVIDENCE
  ------------------------------------------------ ------  --------------------------------
  horizon 1: checkpoint loadable                   PASS    epoch=7 val_loss=0.001992
  horizon 1: checkpoint val_loss matches history   PASS    ckpt=0.001992  history=0.001992
  horizon 2: checkpoint loadable                   PASS    epoch=13 val_loss=0.001629
  horizon 2: checkpoint val_loss matches history   PASS    ckpt=0.001629  history=0.001629
  horizon 3: checkpoint loadable                   PASS    epoch=7 val_loss=0.002674
  horizon 3: checkpoint val_loss matches history   PASS    ckpt=0.002674  history=0.002674
  horizon 4: checkpoint loadable                   PASS    epoch=3 val_loss=0.003132
  horizon 4: checkpoint val_loss matches history   PASS    ckpt=0.003132  history=0.003132
  horizon 5: checkpoint loadable                   PASS    epoch=13 val_loss=0.002580
  horizon 5: checkpoin

---

# Research Phase 2 — Root-Cause-Driven Experiments (WBS, post V1-V5 review)

> Everything from here on is additive: it reuses `GRUModel`, `TrainingConfig`,
> `EarlyStopping`, `train_one_epoch()`, `train_model()`, `build_mmap_dataloaders()`
> unmodified, and writes to separate checkpoint paths so nothing in Phase 3 above
> (the 10 production `gru_horizon_N_best.pt` files) is ever touched by an
> experiment. Each step corresponds 1:1 to a WBS task from the V1-V5 root-cause
> report.

## Step 20: WBS Task 1 — Fix #1 Experiment (CPU Delta Target, Horizon 1)

Tests the highest-confidence root cause from the V4/V5 investigation:
`cpu_usage` is trained as a raw, monotonically-increasing cumulative counter
whose absolute scale varies ~30x between train's cases and val/test's cases.
This cell trains Horizon 1 with `CPU_DELTA_MODE` enabled (defined in the
`MmapSequenceDataset` cell above) and the next cell reconstructs real-unit CPU
metrics for a fair comparison against the existing baseline
(`gru_horizon_1_best.pt`: val MAPE 30.23%, MAE 613.68, RMSE 1443.37).

**Decision rule:** keep only if experiment CPU MAPE < baseline CPU MAPE.
Revert (do nothing — the baseline checkpoint is never touched) otherwise.

## Step 21: WBS Task 2 — Case-Level Heterogeneity Investigation

Investigates the finding from the V1-V5 review that was never previously
examined: test MAPE (254-522%) runs roughly 10x worse than val MAPE (25-45%)
on every horizon, despite `complex_case1` (test) and `single_case1` (val)
showing *similar* raw CPU scale in the earlier distribution-shift diagnostic.
This means the simple train/val scale-mismatch story likely doesn't fully
explain test's performance on its own. This cell compares `complex_case1`
(100% of test) against `complex_case2` (part of train) directly, and looks at
the *distribution* of per-sample error on test, not just its mean.

In [79]:
print("="*70)
print("STEP 21: CASE-LEVEL HETEROGENEITY INVESTIGATION")
print("(complex_case1 [test] vs. complex_case2 [inside train])")
print("="*70)

import numpy as np
import pandas as pd
from pathlib import Path

seq_dir = Path(config.sequences_dir)
TARGET_NAMES_LOCAL = ['cpu_usage', 'mem_usage', 'mem_working_set', 'mem_rss']
TARGET_COLUMNS_LOCAL = [
    'container_cpu_usage_seconds_total', 'container_memory_usage_bytes',
    'container_memory_working_set_bytes', 'container_memory_rss',
]

# --- Part A: isolate complex_case2 from train, if this session's local merged
# CSV from Phase 1 is still present (ephemeral -- wiped on runtime restart).
# Falls back to a clear message (not a silent guess) if unavailable. ---
_default_merged = '/content/processed_data/merged' if globals().get('IN_COLAB') else \
    '/kaggle/working/processed_data/merged' if globals().get('IN_KAGGLE') else '/content/processed_data/merged'
_merged_path = Path(merged_path) if 'merged_path' in dir() else Path(_default_merged)
_train_csv = _merged_path / 'train_data_with_features.csv'

if _train_csv.exists():
    print(f"\nUsing {_train_csv} to isolate complex_case2 from the mixed train split.")
    _train_df = pd.read_csv(_train_csv, usecols=['case_source'] + TARGET_COLUMNS_LOCAL)
    _complex_case2 = _train_df[_train_df['case_source'] == 'complex_case2']
    print(f"complex_case2 rows isolated: {len(_complex_case2):,} (of {len(_train_df):,} total train rows)")
else:
    print(f"\n{_train_csv} not found this session (expected if the runtime/session restarted "
          f"since Phase 1 ran -- local working storage is ephemeral on both Colab and "
          f"Kaggle). Re-run Phase 1's early cells in this session if you need the "
          f"precise complex_case2-only comparison; skipping Part A for now.")
    _complex_case2 = None

# --- Part B: real (inverse-normalized) stats, test (complex_case1) vs.
# complex_case2 (or noted unavailable) ---
y_test = np.load(seq_dir / 'sequences_y_test.npy', mmap_mode='r')
test_real = np.array(y_test[:, 0, :]) * TARGET_STD + TARGET_MEAN

print(f"\n{'Target':<18} {'complex_case1 (test) mean/std':>32} {'complex_case2 mean/std':>28}")
print(f"{'-'*18} {'-'*32} {'-'*28}")
for i, name in enumerate(TARGET_NAMES_LOCAL):
    test_mean, test_std = test_real[:, i].mean(), test_real[:, i].std()
    if _complex_case2 is not None:
        c2_real = _complex_case2[TARGET_COLUMNS_LOCAL[i]].values * TARGET_STD[i] + TARGET_MEAN[i]
        c2_mean, c2_std = c2_real.mean(), c2_real.std()
        print(f"{name:<18} {test_mean:>18.2f} / {test_std:<11.2f} {c2_mean:>14.2f} / {c2_std:<11.2f}")
    else:
        print(f"{name:<18} {test_mean:>18.2f} / {test_std:<11.2f} {'(unavailable this session)':>28}")

# --- Part C: per-sample APE distribution on test, Horizon-1 baseline checkpoint --
# reveals whether a small subset of extreme errors dominates the test MAPE
# figures, or whether error is broadly elevated across all of test. ---
print(f"\n--- Per-sample APE distribution on TEST, Horizon 1 baseline checkpoint ---")
ckpt = torch.load(str(Path(config.checkpoint_dir) / 'gru_horizon_1_best.pt'), map_location=device)
_model = GRUModel(input_size=config.input_size, hidden_size=config.hidden_size,
                   num_layers=config.num_layers, output_size=config.output_size,
                   dropout=config.dropout).to(device)
_model.load_state_dict(ckpt['model_state_dict'])
_model.eval()

test_loader = build_mmap_dataloaders(str(seq_dir), horizon=1, batch_size=config.batch_size)['test']
all_preds, all_targets = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = _model(X_batch)
        all_preds.append(preds.cpu())
        all_targets.append(y_batch)

preds_norm = torch.cat(all_preds, dim=0).numpy()
targets_norm = torch.cat(all_targets, dim=0).numpy()
preds_real = preds_norm * TARGET_STD + TARGET_MEAN
targets_real = targets_norm * TARGET_STD + TARGET_MEAN

epsilon = 1e-6
abs_err = np.abs(preds_real - targets_real)
near_zero = np.abs(targets_real) < epsilon
safe_denom = np.where(near_zero, epsilon, np.abs(targets_real))
ape = np.where(near_zero, np.nan, abs_err / safe_denom * 100.0)

print(f"\n{'Target':<18} {'P50':>10} {'P90':>10} {'P99':>10} {'Max':>12} {'Mean':>10}")
print(f"{'-'*18} {'-'*10} {'-'*10} {'-'*10} {'-'*12} {'-'*10}")
for i, name in enumerate(TARGET_NAMES_LOCAL):
    col = ape[:, i]
    col_valid = col[~np.isnan(col)]
    p50, p90, p99 = np.percentile(col_valid, [50, 90, 99])
    print(f"{name:<18} {p50:>9.1f}% {p90:>9.1f}% {p99:>9.1f}% {col_valid.max():>11.1f}% {col_valid.mean():>9.1f}%")

print(f"\nIf P50 is much lower than the mean, a small tail of extreme errors is")
print(f"driving the average (pathological containers/windows). If P50 is ALSO")
print(f"high, the elevated error is broad across test, not just outlier-driven.")

del _model
torch.cuda.empty_cache(); gc.collect()

STEP 21: CASE-LEVEL HETEROGENEITY INVESTIGATION
(complex_case1 [test] vs. complex_case2 [inside train])

Using /kaggle/working/processed_data/merged/train_data_with_features.csv to isolate complex_case2 from the mixed train split.


2026-07-24 13:15:52,159 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876
2026-07-24 13:15:52,163 - INFO - [mmap] horizon=1 train: 149,916 samples | 2343 batches
2026-07-24 13:15:52,164 - INFO - [mmap] horizon=1 val: 20,348 samples | 318 batches
2026-07-24 13:15:52,166 - INFO - [mmap] horizon=1 test: 20,356 samples | 319 batches


complex_case2 rows isolated: 0 (of 156,666 total train rows)

Target                complex_case1 (test) mean/std       complex_case2 mean/std
------------------ -------------------------------- ----------------------------
cpu_usage                     1319.24 / 1046.91                nan / nan        
mem_usage                 83784176.00 / 13894878.00            nan / nan        
mem_working_set           82703576.00 / 13290282.00            nan / nan        
mem_rss                   70332608.00 / 9577807.00             nan / nan        

--- Per-sample APE distribution on TEST, Horizon 1 baseline checkpoint ---

Target                    P50        P90        P99          Max       Mean
------------------ ---------- ---------- ---------- ------------ ----------
cpu_usage                2.0%       8.1%      12.3%        13.3%       3.0%
mem_usage                0.5%       0.9%       1.4%         3.0%       0.5%
mem_working_set          0.4%       1.1%       1.9%         3.0%       

11

## Step 22: WBS Task 3 — Persistence Baseline, Per Target, All Horizons

Extends the earlier persistence-baseline diagnostic (which only reported an
aggregate 4-target MAPE) to report MAE/RMSE/MAPE **per target**, for every
horizon, alongside the GRU's own per-target numbers. This directly answers
whether the GRU beats a trivial baseline on the 3 memory targets specifically
— not yet checked — as well as confirming CPU's known failure.

In [80]:
print("="*70)
print("STEP 22: PERSISTENCE BASELINE -- PER-TARGET, ALL HORIZONS")
print("="*70)

import json
import numpy as np

seq_dir = Path(config.sequences_dir)
meta = json.load(open(seq_dir / 'sequences_metadata_val.json'))
feature_cols = meta['feature_cols']
TARGET_NAMES_LOCAL = ['cpu_usage', 'mem_usage', 'mem_working_set', 'mem_rss']
TARGET_COLUMNS_LOCAL = [
    'container_cpu_usage_seconds_total', 'container_memory_usage_bytes',
    'container_memory_working_set_bytes', 'container_memory_rss',
]
target_idx_in_X = [feature_cols.index(c) for c in TARGET_COLUMNS_LOCAL]

X_val = np.load(seq_dir / 'sequences_X_val.npy', mmap_mode='r')
y_val = np.load(seq_dir / 'sequences_y_val.npy', mmap_mode='r')

epsilon = 1e-6
print(f"\n{'H':>3} | " + " | ".join(f"{n:>16s}" for n in TARGET_NAMES_LOCAL) + " | GRU beats persistence?")
print("-" * (6 + 19 * 4 + 26))

_ckpt_dir = Path(config.checkpoint_dir)
for h in range(1, 11):
    last_val_norm = np.array(X_val[:, -1, target_idx_in_X])
    target_norm   = np.array(y_val[:, h - 1, :])
    pred_real   = last_val_norm * TARGET_STD + TARGET_MEAN
    target_real = target_norm   * TARGET_STD + TARGET_MEAN

    abs_err = np.abs(pred_real - target_real)
    near_zero = np.abs(target_real) < epsilon
    safe_denom = np.where(near_zero, epsilon, np.abs(target_real))
    ape = np.where(near_zero, np.nan, abs_err / safe_denom * 100.0)
    persistence_mape_per_target = np.nanmean(ape, axis=0)

    _p = _ckpt_dir / f"gru_horizon_{h}_best.pt"
    gru_mape_per_target = [float('nan')] * 4
    beats = []
    if _p.exists():
        ckpt = torch.load(str(_p), map_location='cpu')
        vm = ckpt.get('val_metrics')
        if vm is not None:
            gru_mape_per_target = [vm['mape_per_metric'][n]['mape'] for n in TARGET_NAMES_LOCAL]
            beats = [g < p for g, p in zip(gru_mape_per_target, persistence_mape_per_target)]

    row = f"{h:>3} | " + " | ".join(
        f"P={persistence_mape_per_target[i]:5.1f}% G={gru_mape_per_target[i]:5.1f}%"
        for i in range(4)
    )
    beat_str = "".join("Y" if b else "N" for b in beats) if beats else "????"
    print(f"{row} | {beat_str}")

print(f"\n'P=' is the persistence baseline's MAPE, 'G=' is the GRU's, for each target")
print(f"in order [{', '.join(TARGET_NAMES_LOCAL)}]. The 4-letter Y/N column reports")
print(f"whether the GRU beat persistence on that target -- checked per-target for")
print(f"the first time here, not just as a 4-target average.")

STEP 22: PERSISTENCE BASELINE -- PER-TARGET, ALL HORIZONS

  H |        cpu_usage |        mem_usage |  mem_working_set |          mem_rss | GRU beats persistence?
------------------------------------------------------------------------------------------------------------
  1 | P=  0.0% G=  2.3% | P=  0.1% G=  0.5% | P=  0.1% G=  0.4% | P=  0.1% G=  0.5% | NNNN
  2 | P=  0.0% G=  1.3% | P=  0.1% G=  0.5% | P=  0.1% G=  0.5% | P=  0.2% G=  0.5% | NNNN
  3 | P=  0.0% G=  2.4% | P=  0.2% G=  0.5% | P=  0.2% G=  0.5% | P=  0.2% G=  0.5% | NNNN
  4 | P=  0.1% G=  2.1% | P=  0.3% G=  0.7% | P=  0.3% G=  0.8% | P=  0.3% G=  0.6% | NNNN
  5 | P=  0.1% G=  2.0% | P=  0.3% G=  0.7% | P=  0.3% G=  0.7% | P=  0.4% G=  0.5% | NNNN
  6 | P=  0.1% G=  2.6% | P=  0.3% G=  0.6% | P=  0.3% G=  0.6% | P=  0.4% G=  0.5% | NNNN
  7 | P=  0.1% G=  3.9% | P=  0.4% G=  0.7% | P=  0.4% G=  0.7% | P=  0.4% G=  0.6% | NNNN
  8 | P=  0.1% G=  2.4% | P=  0.4% G=  0.6% | P=  0.4% G=  0.5% | P=  0.4% G=  0.5% | NNNN

## Step 23: WBS Task 4 — Multi-Seed Variance Check (Horizon 1)

Every result in the V1-V5 investigation is a single-seed point estimate. This
cell re-runs Horizon 1 (whichever target framing is currently active via
`CPU_DELTA_MODE`) with 2 additional seeds, to establish whether the ~30% val
MAPE figure already on record is representative or largely seed noise —
needed before treating the per-horizon spread (25-45% across horizons 1-10) as
a real difficulty gradient rather than run-to-run variance.

In [81]:
print("="*70)
print("STEP 23: MULTI-SEED VARIANCE CHECK -- HORIZON 1")
print("="*70)
print(f"CPU_DELTA_MODE is currently: {CPU_DELTA_MODE} "
      f"(this run tests whichever target framing is currently active)")

import time
import numpy as np

SEEDS_TO_TEST = [42, 142, 242]  # base seeds; actual seed used is base+1 (horizon-1 convention)
seq_dir = Path(config.sequences_dir)

_variance_results = []
for base_seed in SEEDS_TO_TEST:
    print(f"\n--- Seed {base_seed + 1} ---")
    set_seed(base_seed + 1)
    loaders = build_mmap_dataloaders(str(seq_dir), horizon=1, batch_size=config.batch_size)
    model = GRUModel(input_size=config.input_size, hidden_size=config.hidden_size,
                      num_layers=config.num_layers, output_size=config.output_size,
                      dropout=config.dropout).to(device)

    # Reuse train_model() completely unmodified -- writes to a seed-specific
    # checkpoint path so the production gru_horizon_1_best.pt is never touched.
    _original_ckpt_dir = config.checkpoint_dir
    config.checkpoint_dir = str(Path(_original_ckpt_dir) / f'seed_variance_check/seed_{base_seed}')
    Path(config.checkpoint_dir).mkdir(parents=True, exist_ok=True)

    start = time.time()
    history, best_val_loss = train_model(model, loaders['train'], loaders['val'], config, horizon=1)
    elapsed = time.time() - start

    ckpt = torch.load(str(Path(config.checkpoint_dir) / 'gru_horizon_1_best.pt'), map_location='cpu')
    val_mape = ckpt['val_metrics']['mape_mean']
    _variance_results.append({'seed': base_seed + 1, 'val_loss': best_val_loss,
                               'val_mape': val_mape, 'time_s': elapsed})
    print(f"  seed={base_seed+1}  best_val_loss={best_val_loss:.6f}  val_MAPE={val_mape:.2f}%  ({elapsed:.0f}s)")

    config.checkpoint_dir = _original_ckpt_dir
    del model, loaders
    torch.cuda.empty_cache(); gc.collect()

_mapes = [r['val_mape'] for r in _variance_results]
_losses = [r['val_loss'] for r in _variance_results]
print(f"\n{'='*70}")
print(f"VARIANCE SUMMARY across {len(SEEDS_TO_TEST)} seeds (Horizon 1)")
print(f"  val_MAPE : mean={np.mean(_mapes):.2f}%  std={np.std(_mapes):.2f}%  "
      f"range=[{min(_mapes):.2f}%, {max(_mapes):.2f}%]")
print(f"  val_loss : mean={np.mean(_losses):.6f}  std={np.std(_losses):.6f}")
print(f"{'='*70}")
print(f"\nIf std is small relative to the mean, the previously-reported single-seed")
print(f"horizon-1 figure is representative, not noise -- the spread seen across")
print(f"all 10 horizons likely reflects real per-horizon difficulty, not seed variance.")

2026-07-24 13:15:53,825 - INFO - [mmap] horizon=1 train: 149,916 samples | 2343 batches
2026-07-24 13:15:53,827 - INFO - [mmap] horizon=1 val: 20,348 samples | 318 batches
2026-07-24 13:15:53,828 - INFO - [mmap] horizon=1 test: 20,356 samples | 319 batches
2026-07-24 13:15:53,837 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876


STEP 23: MULTI-SEED VARIANCE CHECK -- HORIZON 1
CPU_DELTA_MODE is currently: False (this run tests whichever target framing is currently active)

--- Seed 43 ---
  Epoch   1/50 | train=0.020514 | val=0.010332 | val_MAPE=2.68% | lr=1.00e-03 | patience=0/10 | 22s
  Epoch   5/50 | train=0.012240 | val=0.006781 | val_MAPE=1.13% | lr=1.00e-03 | patience=2/10 | 112s
  Epoch  10/50 | train=0.010028 | val=0.004463 | val_MAPE=1.05% | lr=1.00e-03 | patience=3/10 | 224s
  Epoch  15/50 | train=0.007100 | val=0.006550 | val_MAPE=1.28% | lr=5.00e-04 | patience=8/10 | 335s
  Early stopping triggered at epoch 17
  Done: 17 epochs | best_val_loss=0.001992 | 380s
  seed=43  best_val_loss=0.001992  val_MAPE=0.92%  (380s)


2026-07-24 13:22:14,136 - INFO - [mmap] horizon=1 train: 149,916 samples | 2343 batches
2026-07-24 13:22:14,137 - INFO - [mmap] horizon=1 val: 20,348 samples | 318 batches
2026-07-24 13:22:14,139 - INFO - [mmap] horizon=1 test: 20,356 samples | 319 batches
2026-07-24 13:22:14,147 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876



--- Seed 143 ---
  Epoch   1/50 | train=0.021286 | val=0.004061 | val_MAPE=1.45% | lr=1.00e-03 | patience=0/10 | 22s
  Epoch   5/50 | train=0.011861 | val=0.004975 | val_MAPE=1.53% | lr=1.00e-03 | patience=3/10 | 112s
  Epoch  10/50 | train=0.010025 | val=0.004254 | val_MAPE=1.53% | lr=1.00e-03 | patience=3/10 | 223s
  Epoch  15/50 | train=0.006917 | val=0.011181 | val_MAPE=1.80% | lr=5.00e-04 | patience=8/10 | 335s
  Early stopping triggered at epoch 17
  Done: 17 epochs | best_val_loss=0.001851 | 380s
  seed=143  best_val_loss=0.001851  val_MAPE=0.97%  (380s)


2026-07-24 13:28:34,198 - INFO - [mmap] horizon=1 train: 149,916 samples | 2343 batches
2026-07-24 13:28:34,200 - INFO - [mmap] horizon=1 val: 20,348 samples | 318 batches
2026-07-24 13:28:34,202 - INFO - [mmap] horizon=1 test: 20,356 samples | 319 batches
2026-07-24 13:28:34,210 - INFO - [GRUModel] input=27 hidden=128 layers=2 output=4 dropout=0.2 | params=167,876



--- Seed 243 ---
  Epoch   1/50 | train=0.020467 | val=0.002617 | val_MAPE=1.66% | lr=1.00e-03 | patience=0/10 | 22s
  Epoch   5/50 | train=0.011843 | val=0.003037 | val_MAPE=1.11% | lr=1.00e-03 | patience=4/10 | 112s
  Epoch  10/50 | train=0.008261 | val=0.002656 | val_MAPE=1.04% | lr=5.00e-04 | patience=2/10 | 223s
  Epoch  15/50 | train=0.006225 | val=0.004690 | val_MAPE=1.42% | lr=2.50e-04 | patience=7/10 | 335s
  Early stopping triggered at epoch 18
  Done: 18 epochs | best_val_loss=0.002350 | 402s
  seed=243  best_val_loss=0.002350  val_MAPE=0.89%  (402s)

VARIANCE SUMMARY across 3 seeds (Horizon 1)
  val_MAPE : mean=0.93%  std=0.03%  range=[0.89%, 0.97%]
  val_loss : mean=0.002064  std=0.000210

If std is small relative to the mean, the previously-reported single-seed
horizon-1 figure is representative, not noise -- the spread seen across
all 10 horizons likely reflects real per-horizon difficulty, not seed variance.
